In [1]:
"""
MODULE 9 — PORTFOLIO OPTIMIZATION, CAPITAL ALLOCATION &
           STRATEGIC LENDING INTELLIGENCE PLATFORM
 
Author : Senior Quantitative Risk Strategist & Portfolio Optimization Architect
Depends: Modules 1–8 outputs
Primary Input: lending_features/master_feature_table.csv + all module outputs
Outputs: portfolio_optimization/ — optimization, simulations, dashboards, reports
 
Steps:
  1  Portfolio Strategy Framework
  2  Portfolio Risk Measurement
  3  Profitability Optimization
  4  Capital Allocation Engine
  5  Exposure Optimization
  6  Segment-Level Portfolio Optimization
  7  Risk-Return Optimization (Efficient Frontier)
  8  Approval Strategy Optimization
  9  Pricing-Portfolio Integration
  10 Collections-Portfolio Integration
  11 Scenario Simulation Engine
  12 Stress Testing Framework
  13 Concentration Risk Management
  14 Dynamic Portfolio Rebalancing
  15 Executive Portfolio Dashboards
  16 Advanced Visual Analytics
  17 Governance & Risk Appetite Framework
  18 Strategic Recommendation Engine
  19 Production Portfolio Intelligence Pipeline
"""

'\nMODULE 9 — PORTFOLIO OPTIMIZATION, CAPITAL ALLOCATION &\n           STRATEGIC LENDING INTELLIGENCE PLATFORM\n\nAuthor : Senior Quantitative Risk Strategist & Portfolio Optimization Architect\nDepends: Modules 1–8 outputs\nPrimary Input: lending_features/master_feature_table.csv + all module outputs\nOutputs: portfolio_optimization/ — optimization, simulations, dashboards, reports\n\nSteps:\n  1  Portfolio Strategy Framework\n  2  Portfolio Risk Measurement\n  3  Profitability Optimization\n  4  Capital Allocation Engine\n  5  Exposure Optimization\n  6  Segment-Level Portfolio Optimization\n  7  Risk-Return Optimization (Efficient Frontier)\n  8  Approval Strategy Optimization\n  9  Pricing-Portfolio Integration\n  10 Collections-Portfolio Integration\n  11 Scenario Simulation Engine\n  12 Stress Testing Framework\n  13 Concentration Risk Management\n  14 Dynamic Portfolio Rebalancing\n  15 Executive Portfolio Dashboards\n  16 Advanced Visual Analytics\n  17 Governance & Risk Appeti

In [5]:
# =============================================================================
# CELL 1 — Install dependencies
# =============================================================================
!pip install pandas numpy scipy scikit-learn matplotlib seaborn plotly \
             cvxpy pyportfolioopt optuna joblib statsmodels xgboost lightgbm
 

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/887.3 kB ? eta -:--:--
   --------------------------------------- 887.3/887.3 kB 13.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 16.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ----------- ---------------------------- 2.1/7.5 MB 14.7 MB/s eta 0:00:01
   ----------- ---------------------------- 2.1/7.5 MB 14.7 MB/s eta 0:00:01
   ---------------- ----------------------- 3.1/7.5 MB 5.4 MB/s eta 0:00:01
   ---------------------- ----------------- 4.2/7.5 MB 5.9 MB/s eta 0:00:01
   --------------------------


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [6]:
# =============================================================================
# CELL 2 — Imports & Global Configuration
# =============================================================================
import os, sys, json, warnings, logging, joblib, time
from pathlib import Path
from datetime import datetime
from copy import deepcopy
 
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, differential_evolution
from scipy.stats import ks_2samp
 
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
 
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
 
try:
    import cvxpy as cp
    CVXPY_OK = True
except ImportError:
    CVXPY_OK = False
    print("⚠  cvxpy not found — analytical optimization fallbacks active")
 
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    OPTUNA_OK = False
 
warnings.filterwarnings("ignore")
try:
    get_ipython()
    matplotlib.use("inline")
except NameError:
    matplotlib.use("Agg")
 
SEED = 42
np.random.seed(SEED)
 
# ── Palette ────────────────────────────────────────────────────────────────
PAL = {
    "primary":   "#2C5F8A",
    "danger":    "#D94F3D",
    "success":   "#2E8B57",
    "warning":   "#E8A838",
    "neutral":   "#6B7280",
    "highlight": "#7B2D8B",
    "orange":    "#E07B39",
    "bg":        "#F8F9FA",
    "teal":      "#1ABC9C",
    "gold":      "#C9A227",
}
GRADE_COLORS = {
    "A": "#2E8B57", "B": "#7DCE82",
    "C": "#E8A838", "D": "#E07B39", "E": "#D94F3D",
}
HEALTH_COLORS = {
    "Green": "#2E8B57", "Yellow": "#E8A838",
    "Orange": "#E07B39", "Red": "#D94F3D",
}
 
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": PAL["bg"],
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.titleweight": "bold",
    "axes.titlesize":   12,
})
 
# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR  = r"C:\Users\abhir\OneDrive\Desktop\iitg"
FEAT_DIR  = os.path.join(BASE_DIR, "lending_features")
PO_DIR    = os.path.join(BASE_DIR, "portfolio_optimization")
OPT_DIR   = os.path.join(PO_DIR,  "optimization")
SIM_DIR   = os.path.join(PO_DIR,  "simulations")
DASH_DIR  = os.path.join(PO_DIR,  "dashboards")
RPT_DIR   = os.path.join(PO_DIR,  "reports")
GOV_DIR   = os.path.join(PO_DIR,  "governance")
STR_DIR   = os.path.join(PO_DIR,  "stress_testing")
MON_DIR   = os.path.join(PO_DIR,  "monitoring")
PIP_DIR   = os.path.join(PO_DIR,  "pipelines")
NB_DIR    = os.path.join(PO_DIR,  "notebooks")
 
for d in [PO_DIR, OPT_DIR, SIM_DIR, DASH_DIR, RPT_DIR,
          GOV_DIR, STR_DIR, MON_DIR, PIP_DIR, NB_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
 
# ── Logging ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(
            os.path.join(PO_DIR, "module9.log"), mode="w", encoding="utf-8"
        ),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("PortfolioOpt")
 
def savefig(name, dpi=150, folder=DASH_DIR):
    path = os.path.join(folder, name)
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=PAL["bg"])
    plt.close()
    log.info("  Figure: %s", name)
    return path
 
def section(title):
    bar = "═" * 70
    print(f"\n{bar}\n  {title}\n{bar}")
 
def get_col(df, col, default):
    return df[col].fillna(default).values if col in df.columns \
           else np.full(len(df), default)
 
log.info("Module 9 — Portfolio Optimization platform started.")
print("✅  Module 9 configuration loaded. All portfolio_optimization/ dirs ready.")
 

10:27:00 | INFO     | Module 9 — Portfolio Optimization platform started.
✅  Module 9 configuration loaded. All portfolio_optimization/ dirs ready.


In [7]:
# =============================================================================
# CELL 3 — STEP 1: Portfolio Strategy Framework
# =============================================================================
 
section("STEP 1 — PORTFOLIO STRATEGY FRAMEWORK")
 
PORTFOLIO_STRATEGY_DOC = """
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO OPTIMIZATION STRATEGY FRAMEWORK               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHY PORTFOLIO OPTIMIZATION IN LENDING?                              ║
║  Individual loan decisions are necessary but insufficient.           ║
║  Portfolio-level optimization asks: GIVEN all loans we can make,    ║
║  which combination maximizes risk-adjusted return subject to:        ║
║  • Capital constraints   (regulatory + economic capital)             ║
║  • Risk appetite limits  (NPA ceiling, EL ceiling)                   ║
║  • Concentration limits  (single segment, geography, product)        ║
║  • Growth constraints    (disbursement targets, approval floors)     ║
║  • Operational capacity  (collections bandwidth, servicing cost)     ║
║                                                                      ║
║  PORTFOLIO OPTIMIZATION OBJECTIVES:                                  ║
║  1. Maximize risk-adjusted net interest margin (NIM)                 ║
║  2. Minimize expected credit loss (ECL) as % of AUM                 ║
║  3. Maximize RAROC across the portfolio book                         ║
║  4. Optimize capital utilization (ROE, ROA)                          ║
║  5. Control NPA formation rate below regulatory thresholds           ║
║  6. Diversify concentration across segments, products, geographies   ║
║  7. Balance short-term profitability vs long-term portfolio quality  ║
║                                                                      ║
║  RISK APPETITE FRAMEWORK:                                            ║
║  • Max portfolio PD                 : 8.0%                           ║
║  • Max NPA ratio                    : 3.0%                           ║
║  • Max Grade E exposure             : 10% of book                    ║
║  • Min RAROC                        : 0.80                           ║
║  • Max segment concentration        : 40% of book                    ║
║  • Max product concentration        : 50% of book                    ║
║  • Max geographic concentration     : 35% of book                    ║
║  • Min collections cure rate        : 45%                            ║
║  • Max EL / Gross Interest          : 35%                            ║
║                                                                      ║
║  GROWTH vs RISK TRADEOFF:                                            ║
║  Fintech lenders face a fundamental tension:                         ║
║  ↑ Approval rate → ↑ Volume → ↑ NPA (non-linear at tails)           ║
║  ↓ Approval rate → ↓ Volume → ↓ NPA → ↓ Revenue                    ║
║  Optimal = approval threshold that maximizes risk-adjusted profit    ║
║                                                                      ║
║  PORTFOLIO CONSTRUCTION PHILOSOPHY:                                  ║
║  • Grade A/B: Profitability drivers — scale aggressively             ║
║  • Grade C:   Balance growth and risk — selective expansion          ║
║  • Grade D:   Collateral/co-borrower required — limit concentration  ║
║  • Grade E:   Decline / redirect — do not accumulate                 ║
║  • Collections-aware: higher weight to borrowers with cure potential ║
╚══════════════════════════════════════════════════════════════════════╝
"""
 
print(PORTFOLIO_STRATEGY_DOC)
 
# ── Risk Appetite Framework ────────────────────────────────────────────────
RISK_APPETITE = {
    "max_portfolio_pd":           0.08,
    "max_npa_ratio":              0.030,
    "max_grade_e_pct":            0.10,
    "min_raroc":                  0.80,
    "max_segment_concentration":  0.40,
    "max_product_concentration":  0.50,
    "max_geo_concentration":      0.35,
    "min_cure_rate":              0.45,
    "max_el_to_interest":         0.35,
    "min_approval_rate":          0.60,
    "max_approval_rate":          0.95,
    "target_nim":                 0.06,       # 6% NIM target
    "capital_ratio":              0.08,       # 8% capital requirement
    "max_ead_per_borrower":       5_000_000,  # ₹50 lakh per borrower
    "max_single_sector_pct":      0.40,
}
 
# ── Portfolio Construction Targets ────────────────────────────────────────
PORTFOLIO_TARGETS = {
    "A": {"min_pct": 0.05, "max_pct": 0.25, "target_pct": 0.15,
          "priority": "High — prime quality anchor"},
    "B": {"min_pct": 0.15, "max_pct": 0.35, "target_pct": 0.25,
          "priority": "High — near-prime volume driver"},
    "C": {"min_pct": 0.20, "max_pct": 0.40, "target_pct": 0.35,
          "priority": "Medium — selective with premium pricing"},
    "D": {"min_pct": 0.05, "max_pct": 0.15, "target_pct": 0.10,
          "priority": "Low — collateral/co-borrower required"},
    "E": {"min_pct": 0.00, "max_pct": 0.10, "target_pct": 0.05,
          "priority": "Minimal — EL exceeds risk tolerance"},
}
 
with open(os.path.join(GOV_DIR, "risk_appetite_framework.json"), "w") as f:
    json.dump(RISK_APPETITE, f, indent=2)
with open(os.path.join(GOV_DIR, "portfolio_targets.json"), "w") as f:
    json.dump(PORTFOLIO_TARGETS, f, indent=2)
with open(os.path.join(RPT_DIR, "portfolio_strategy.txt"), "w", encoding="utf-8") as f:
    f.write(PORTFOLIO_STRATEGY_DOC)
 
print("  ✅  Portfolio strategy framework and risk appetite saved.")
log.info("Portfolio strategy framework complete.")


══════════════════════════════════════════════════════════════════════
  STEP 1 — PORTFOLIO STRATEGY FRAMEWORK
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO OPTIMIZATION STRATEGY FRAMEWORK               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHY PORTFOLIO OPTIMIZATION IN LENDING?                              ║
║  Individual loan decisions are necessary but insufficient.           ║
║  Portfolio-level optimization asks: GIVEN all loans we can make,    ║
║  which combination maximizes risk-adjusted return subject to:        ║
║  • Capital constraints   (regulatory + economic capital)             ║
║  • Risk appetite limits  (NPA ceiling, EL ceiling)                   ║
║  • Concentration limits  (single segment, geography, product)        ║
║  • Growth con

In [8]:
# =============================================================================
# CELL 4 — Data Loading & Portfolio Preparation
# =============================================================================
 
section("DATA LOADING & PORTFOLIO PREPARATION")
 
path = os.path.join(FEAT_DIR, "master_feature_table.csv")
if not os.path.exists(path):
    raise FileNotFoundError(f"master_feature_table.csv not found. Run Module 2 first.")
 
df_raw = pd.read_csv(path, low_memory=False)
log.info("Master table loaded: %s rows × %s cols", f"{len(df_raw):,}", df_raw.shape[1])
df = df_raw.copy()
 
# Work with approved loans only
approved = df[df["approval_status"] == "Approved"].copy().reset_index(drop=True) \
    if "approval_status" in df.columns else df.copy()
print(f"  Total portfolio: {len(approved):,} loans")
 
# ── Ensure key columns exist ─────────────────────────────────────────────
def ensure_col(df_in, col, default_fn):
    if col not in df_in.columns:
        df_in[col] = default_fn(df_in)
    return df_in
 
cs = approved["credit_score"].fillna(650) if "credit_score" in approved.columns \
     else pd.Series([650] * len(approved))
 
# PD Score
if "pd_score" not in approved.columns:
    approved["pd_score"] = np.clip(1 - (cs - 300) / 600, 0.02, 0.80)
 
# Risk grade
if "risk_grade" not in approved.columns:
    def _grade(pd_val):
        if pd_val < 0.05: return "A"
        if pd_val < 0.10: return "B"
        if pd_val < 0.18: return "C"
        if pd_val < 0.30: return "D"
        return "E"
    approved["risk_grade"] = approved["pd_score"].apply(_grade)
 
approved["risk_grade_clean"] = approved["risk_grade"].astype(str).str[0].str.upper()
approved["risk_grade_clean"] = approved["risk_grade_clean"].where(
    approved["risk_grade_clean"].isin(["A","B","C","D","E"]), "C"
)
 
# Loan amount
if "loan_amount" not in approved.columns:
    approved["loan_amount"] = np.random.uniform(50000, 500000, len(approved))
 
# Annual income
if "annual_income" not in approved.columns:
    approved["annual_income"] = np.random.uniform(200000, 1200000, len(approved))
 
# LGD by grade
LGD_MAP = {"A": 0.50, "B": 0.55, "C": 0.60, "D": 0.65, "E": 0.70}
approved["lgd"] = approved["risk_grade_clean"].map(LGD_MAP).fillna(0.60)
 
# Expected Loss
if "loan_tenure_months" not in approved.columns:
    approved["loan_tenure_months"] = 24
tenure_yrs = approved["loan_tenure_months"].fillna(24) / 12
approved["cum_pd"] = 1 - (1 - approved["pd_score"]) ** tenure_yrs
approved["expected_loss"] = (
    approved["cum_pd"] * approved["lgd"] * approved["loan_amount"]
).round(2)
 
# Interest rate
if "interest_rate" not in approved.columns:
    approved["interest_rate"] = (
        8.5 + 2.0 + approved["pd_score"] * 20 + 2.0
    ).clip(10, 36)
 
# Gross interest
mr = approved["interest_rate"] / 100 / 12
n  = approved["loan_tenure_months"].fillna(24).astype(int)
emi = approved["loan_amount"] * mr * (1 + mr)**n / ((1 + mr)**n - 1)
approved["emi"] = emi.round(2)
approved["gross_interest"] = (emi * n - approved["loan_amount"]).round(2)
 
# Profitability columns
if "net_income" not in approved.columns:
    SERVICING_PA = 500
    DEFAULT_CAC  = 1000
    approved["net_income"] = (
        approved["gross_interest"]
        - approved["expected_loss"]
        - SERVICING_PA * tenure_yrs
        - DEFAULT_CAC
    ).round(2)
 
# RAROC
approved["raroc"] = (
    approved["net_income"] / (approved["expected_loss"] * 12.5 * 0.08).clip(lower=1)
).round(4)
 
# Default flag
if "default_flag" not in approved.columns:
    approved["default_flag"] = (
        approved["pd_score"] > 0.30
    ).astype(int)
 
# EAD (Exposure at Default)
approved["ead"] = approved["loan_amount"].copy()
 
# Delinquency risk (from Module 7 proxy)
if "delinquency_risk_prob" not in approved.columns:
    approved["delinquency_risk_prob"] = approved["pd_score"] * 0.85
 
# Recovery probability
if "recovery_probability" not in approved.columns:
    approved["recovery_probability"] = np.clip(
        0.65 - approved["pd_score"] * 0.5, 0.10, 0.80
    )
 
# Loan type
if "loan_type" not in approved.columns:
    loan_types = ["Personal Loan", "BNPL", "SME Working Capital"]
    approved["loan_type"] = np.random.choice(loan_types, len(approved),
                                              p=[0.55, 0.20, 0.25])
 
# Employment type
if "employment_type" not in approved.columns:
    emp_types = ["Salaried", "Self-Employed", "Gig Worker", "SME Owner",
                 "First-Time Borrower"]
    approved["employment_type"] = np.random.choice(emp_types, len(approved),
                                                   p=[0.43, 0.20, 0.13, 0.10, 0.14])
 
# Urban flag
if "urban_semiurban_flag" not in approved.columns:
    approved["urban_semiurban_flag"] = np.random.choice(
        ["Urban", "Semi-Urban"], len(approved), p=[0.61, 0.39]
    )
 
# Acquisition channel
if "acquisition_channel" not in approved.columns:
    channels = ["Organic Search", "App Store", "Referral",
                "Social Media", "DSA Partner"]
    approved["acquisition_channel"] = np.random.choice(
        channels, len(approved), p=[0.25, 0.20, 0.20, 0.20, 0.15]
    )
 
# CLV
if "customer_lifetime_value" not in approved.columns:
    approved["customer_lifetime_value"] = (
        approved["gross_interest"] * np.random.uniform(0.5, 2.5, len(approved))
    ).round(0)
 
# Health score proxy
if "borrower_health_score" not in approved.columns:
    approved["borrower_health_score"] = np.clip(
        100 - approved["pd_score"] * 100
        - approved.get("financial_stress_index", pd.Series(0.3, index=approved.index)).fillna(0.3) * 20,
        10, 95
    ).round(1)
 
# NIM per loan
approved["nim_pct"] = (
    (approved["gross_interest"] - approved["expected_loss"])
    / approved["loan_amount"]
    / (tenure_yrs + 1e-6) * 100
).round(3)
 
# Recovery-adjusted profitability
approved["recovery_adj_profit"] = (
    approved["net_income"]
    + approved["expected_loss"] * approved["recovery_probability"]
).round(2)
 
print(f"  Approved loans     : {len(approved):,}")
print(f"  Avg PD             : {approved['pd_score'].mean():.3%}")
print(f"  Avg EL             : ₹{approved['expected_loss'].mean():,.0f}")
print(f"  Avg RAROC          : {approved['raroc'].mean():.3f}")
print(f"  Total AUM          : ₹{approved['loan_amount'].sum()/1e7:.2f} Cr")
print(f"  Total ECL          : ₹{approved['expected_loss'].sum()/1e7:.2f} Cr")
print(f"  ECL/AUM            : {approved['expected_loss'].sum()/approved['loan_amount'].sum():.2%}")
log.info("Data loading and portfolio preparation complete.")


══════════════════════════════════════════════════════════════════════
  DATA LOADING & PORTFOLIO PREPARATION
══════════════════════════════════════════════════════════════════════
10:27:32 | INFO     | Master table loaded: 65,000 rows × 76 cols
  Total portfolio: 24,599 loans
  Approved loans     : 24,599
  Avg PD             : 55.935%
  Avg EL             : ₹194,504
  Avg RAROC          : -0.618
  Total AUM          : ₹807.12 Cr
  Total ECL          : ₹478.46 Cr
  ECL/AUM            : 59.28%
10:27:32 | INFO     | Data loading and portfolio preparation complete.


In [9]:
# =============================================================================
# CELL 5 — STEP 2: Portfolio Risk Measurement
# =============================================================================
 
section("STEP 2 — PORTFOLIO RISK MEASUREMENT")
 
# ── Portfolio-Level Risk Metrics ──────────────────────────────────────────
def compute_portfolio_risk_metrics(df_in: pd.DataFrame) -> dict:
    """
    Compute comprehensive portfolio-level risk metrics.
    These are the core inputs to all optimization algorithms.
    """
    n        = len(df_in)
    aum      = df_in["loan_amount"].sum()
    ecl      = df_in["expected_loss"].sum()
    gross_int= df_in["gross_interest"].sum()
    net_inc  = df_in["net_income"].sum()
    avg_pd   = df_in["pd_score"].mean()
    wt_pd    = (df_in["pd_score"] * df_in["loan_amount"]).sum() / max(aum, 1)
 
    # Herfindahl-Hirschman Index (HHI) — concentration
    grade_shares = (
        df_in.groupby("risk_grade_clean")["loan_amount"].sum() / aum
    ).fillna(0)
    hhi_grade = (grade_shares ** 2).sum()
 
    # Geographic HHI
    if "urban_semiurban_flag" in df_in.columns:
        geo_shares  = df_in.groupby("urban_semiurban_flag")["loan_amount"].sum() / aum
        hhi_geo     = (geo_shares ** 2).sum()
    else:
        hhi_geo = 0.5
 
    # Product HHI
    if "loan_type" in df_in.columns:
        prod_shares = df_in.groupby("loan_type")["loan_amount"].sum() / aum
        hhi_product = (prod_shares ** 2).sum()
    else:
        hhi_product = 0.5
 
    # Default rate
    actual_dr = df_in["default_flag"].mean() if "default_flag" in df_in.columns else avg_pd
 
    # Delinquency exposure
    delq_exposure = (
        df_in[df_in["delinquency_risk_prob"] > 0.50]["loan_amount"].sum()
        if "delinquency_risk_prob" in df_in.columns else 0
    )
 
    # NIM
    nim = (gross_int - ecl) / max(aum, 1) * 100
 
    # Stress VaR (99th pct of EL distribution if we simulate 1000 portfolios)
    np.random.seed(SEED)
    el_bootstrap = [
        (df_in["pd_score"] * np.random.uniform(0.9, 1.15, n) *
         df_in["lgd"] * df_in["loan_amount"]).sum()
        for _ in range(200)
    ]
    var_99 = np.percentile(el_bootstrap, 99)
    cvar_99 = np.mean([x for x in el_bootstrap if x >= var_99])
 
    # Portfolio quality score (0-100)
    pq_score = (
        (1 - wt_pd / 0.30) * 30          # PD component
        + (1 - ecl / max(aum * 0.15, 1)) * 30  # ECL component
        + (1 - hhi_grade) * 20             # Diversification
        + max(0, net_inc / max(gross_int, 1)) * 20  # Profitability
    ) * 100 / 100
    pq_score = float(np.clip(pq_score, 0, 100))
 
    return {
        "n_loans":              n,
        "aum_cr":               round(aum / 1e7, 2),
        "ecl_cr":               round(ecl / 1e7, 2),
        "ecl_to_aum_pct":       round(ecl / max(aum, 1) * 100, 3),
        "gross_interest_cr":    round(gross_int / 1e7, 2),
        "net_income_cr":        round(net_inc / 1e7, 2),
        "avg_pd":               round(avg_pd, 4),
        "wt_avg_pd":            round(wt_pd, 4),
        "actual_default_rate":  round(actual_dr, 4),
        "hhi_grade":            round(float(hhi_grade), 4),
        "hhi_geo":              round(float(hhi_geo), 4),
        "hhi_product":          round(float(hhi_product), 4),
        "nim_pct":              round(nim, 3),
        "avg_raroc":            round(df_in["raroc"].mean(), 4),
        "delq_exposure_cr":     round(delq_exposure / 1e7, 2),
        "var_99_cr":            round(var_99 / 1e7, 2),
        "cvar_99_cr":           round(cvar_99 / 1e7, 2),
        "portfolio_quality_score": round(pq_score, 1),
        "ecl_cover_ratio":      round(gross_int / max(ecl, 1), 3),
    }
 
 
port_risk = compute_portfolio_risk_metrics(approved)
print("\n  Portfolio Risk Snapshot:")
for k, v in port_risk.items():
    print(f"    {k:<35}: {v}")
 
# Grade-level risk decomposition
grade_risk = approved.groupby("risk_grade_clean").agg(
    count        = ("loan_amount",     "count"),
    aum_cr       = ("loan_amount",     lambda x: x.sum()/1e7),
    ecl_cr       = ("expected_loss",   lambda x: x.sum()/1e7),
    avg_pd       = ("pd_score",        "mean"),
    avg_lgd      = ("lgd",             "mean"),
    avg_raroc    = ("raroc",           "mean"),
    net_income_cr= ("net_income",      lambda x: x.sum()/1e7),
    default_rate = ("default_flag",    "mean"),
).reset_index()
grade_risk["exposure_pct"] = (
    grade_risk["aum_cr"] / grade_risk["aum_cr"].sum() * 100
).round(1)
grade_risk["ecl_pct"]      = (
    grade_risk["ecl_cr"] / grade_risk["ecl_cr"].sum() * 100
).round(1)
grade_risk["el_to_aum"]    = (
    grade_risk["ecl_cr"] / grade_risk["aum_cr"] * 100
).round(2)
 
print("\n  Grade-Level Risk Decomposition:")
print(grade_risk[[
    "risk_grade_clean", "count", "exposure_pct", "avg_pd",
    "el_to_aum", "avg_raroc", "ecl_pct"
]].to_string(index=False))
grade_risk.to_csv(os.path.join(RPT_DIR, "grade_risk_decomposition.csv"), index=False)
 
# Risk ladder flags
for _, row in grade_risk.iterrows():
    flags = []
    if row["exposure_pct"] > 40: flags.append("⚠ Concentration >40%")
    if row["avg_pd"] > RISK_APPETITE["max_portfolio_pd"]: flags.append("⚠ PD exceeds appetite")
    if row["avg_raroc"] < RISK_APPETITE["min_raroc"]: flags.append("⚠ RAROC below target")
    if flags:
        print(f"  Grade {row['risk_grade_clean']}: {' | '.join(flags)}")
 
with open(os.path.join(MON_DIR, "portfolio_risk_metrics.json"), "w") as f:
    json.dump(port_risk, f, indent=2)
 
log.info("Portfolio risk measurement complete. Quality score: %.1f", port_risk["portfolio_quality_score"])


══════════════════════════════════════════════════════════════════════
  STEP 2 — PORTFOLIO RISK MEASUREMENT
══════════════════════════════════════════════════════════════════════

  Portfolio Risk Snapshot:
    n_loans                            : 24599
    aum_cr                             : 807.12
    ecl_cr                             : 478.46
    ecl_to_aum_pct                     : 59.28
    gross_interest_cr                  : 414.0
    net_income_cr                      : -69.97
    avg_pd                             : 0.5593
    wt_avg_pd                          : 0.5509
    actual_default_rate                : 0.0194
    hhi_grade                          : 0.9886
    hhi_geo                            : 0.5289
    hhi_product                        : 0.5023
    nim_pct                            : -7.987
    avg_raroc                          : -0.6184
    delq_exposure_cr                   : 300.98
    var_99_cr                          : 319.46
    cvar_99_cr           

In [10]:
# =============================================================================
# CELL 6 — STEP 3: Profitability Optimization
# =============================================================================
 
section("STEP 3 — PROFITABILITY OPTIMIZATION")
 
# ── Segment profitability analysis ────────────────────────────────────────
def compute_segment_profitability(df_in, seg_col):
    """Compute full P&L for each segment group."""
    if seg_col not in df_in.columns:
        return pd.DataFrame()
 
    seg = df_in.groupby(seg_col).agg(
        count            = ("loan_amount",        "count"),
        aum_cr           = ("loan_amount",         lambda x: x.sum()/1e7),
        gross_int_cr     = ("gross_interest",      lambda x: x.sum()/1e7),
        ecl_cr           = ("expected_loss",       lambda x: x.sum()/1e7),
        net_income_cr    = ("net_income",          lambda x: x.sum()/1e7),
        avg_raroc        = ("raroc",               "mean"),
        avg_nim          = ("nim_pct",             "mean"),
        avg_pd           = ("pd_score",            "mean"),
        avg_rate         = ("interest_rate",       "mean"),
        recovery_adj_cr  = ("recovery_adj_profit", lambda x: x.sum()/1e7),
        pct_profitable   = ("net_income",          lambda x: (x>0).mean()*100),
    ).reset_index()
 
    seg["el_ratio"]     = (seg["ecl_cr"] / seg["gross_int_cr"].replace(0, np.nan)).round(3)
    seg["ni_per_loan"]  = (seg["net_income_cr"] * 1e7 / seg["count"].replace(0,1)).round(0)
    seg["profit_rank"]  = seg["avg_raroc"].rank(ascending=False).astype(int)
    return seg
 
 
profitability_segments = {}
for seg_col in ["risk_grade_clean", "loan_type", "employment_type",
                "acquisition_channel"]:
    seg_df = compute_segment_profitability(approved, seg_col)
    if len(seg_df):
        profitability_segments[seg_col] = seg_df
        seg_df.to_csv(os.path.join(RPT_DIR, f"profitability_{seg_col}.csv"), index=False)
        print(f"\n  Profitability by {seg_col}:")
        print(seg_df[[seg_col, "count", "aum_cr", "avg_pd",
                       "avg_raroc", "avg_nim", "pct_profitable",
                       "net_income_cr"]].to_string(index=False))
 
# ── Portfolio yield optimization curve ────────────────────────────────────
pd_thresholds = np.linspace(0.05, 0.80, 30)
yield_curve = []
 
for pd_thresh in pd_thresholds:
    subset = approved[approved["pd_score"] <= pd_thresh]
    if len(subset) < 50:
        continue
    yield_curve.append({
        "pd_ceiling":     round(pd_thresh, 3),
        "n_loans":        len(subset),
        "approval_rate":  round(len(subset) / len(approved), 3),
        "aum_cr":         round(subset["loan_amount"].sum() / 1e7, 2),
        "nim_pct":        round(subset["nim_pct"].mean(), 3),
        "avg_raroc":      round(subset["raroc"].mean(), 4),
        "ecl_to_aum":     round(subset["expected_loss"].sum()
                               / subset["loan_amount"].sum() * 100, 3),
        "net_income_cr":  round(subset["net_income"].sum() / 1e7, 2),
    })
 
yield_df = pd.DataFrame(yield_curve)
yield_df.to_csv(os.path.join(OPT_DIR, "portfolio_yield_curve.csv"), index=False)
log.info("Profitability optimization complete.")
print("\n  ✅  Profitability analysis and yield curves saved.")


══════════════════════════════════════════════════════════════════════
  STEP 3 — PROFITABILITY OPTIMIZATION
══════════════════════════════════════════════════════════════════════

  Profitability by risk_grade_clean:
risk_grade_clean  count     aum_cr   avg_pd  avg_raroc    avg_nim  pct_profitable  net_income_cr
               B    121   3.970832 0.053333  -1.859598   3.413702       78.512397       0.311744
               D     18   0.667284 0.286296  -1.212828  -6.321667        5.555556      -0.091677
               E  24460 802.485477 0.562053  -0.611865 -13.676714       15.502862     -70.193148

  Profitability by loan_type:
          loan_type  count     aum_cr   avg_pd  avg_raroc    avg_nim  pct_profitable  net_income_cr
               BNPL   5166   7.051805 0.560326  -1.911953 -32.678432        0.000000      -1.162117
      Personal Loan  13742 340.523132 0.558281  -0.357064 -10.361259       12.196187     -57.162919
SME Working Capital   5691 459.548655 0.561040  -0.075408  -4.

In [11]:
# =============================================================================
# CELL 7 — STEP 4: Capital Allocation Engine
# =============================================================================
 
section("STEP 4 — CAPITAL ALLOCATION ENGINE")
 
# ─────────────────────────────────────────────────────────────────────────
# CAPITAL ALLOCATION FRAMEWORK:
# Basel III / RBI mandate banks to hold capital against credit risk.
# For unsecured retail lending: RWA = EAD × Risk Weight × Capital Ratio
# Risk Weight varies by PD, LGD, correlation (IRB approach).
# Simplified: Risk Weight ≈ PD^0.5 × LGD × 12.5 (standardised)
#
# Capital Allocation Optimization:
# Given a fixed capital budget K, allocate to maximize portfolio NI
# subject to:
#   sum(capital_i) ≤ K  (capital constraint)
#   sum(loan_i × PD_i × LGD_i) ≤ EL_ceiling  (risk constraint)
#   Concentration limits per segment
# ─────────────────────────────────────────────────────────────────────────
 
TOTAL_CAPITAL_CR = 250.0   # ₹250 crore capital budget
CAPITAL_RATIO    = 0.08    # 8% minimum capital ratio
 
# Economic capital per loan
approved["economic_capital"] = (
    approved["expected_loss"] * 12.5 * CAPITAL_RATIO
).round(2)
 
approved["capital_efficiency"] = (
    approved["net_income"] / approved["economic_capital"].clip(lower=1)
).round(4)
 
# Capital allocation by grade
cap_alloc = approved.groupby("risk_grade_clean").agg(
    count              = ("loan_amount",         "count"),
    aum_cr             = ("loan_amount",          lambda x: x.sum()/1e7),
    ecl_cr             = ("expected_loss",        lambda x: x.sum()/1e7),
    eco_cap_cr         = ("economic_capital",     lambda x: x.sum()/1e7),
    net_income_cr      = ("net_income",           lambda x: x.sum()/1e7),
    avg_cap_efficiency = ("capital_efficiency",   "mean"),
    avg_raroc          = ("raroc",                "mean"),
).reset_index()
cap_alloc["cap_pct"]  = (cap_alloc["eco_cap_cr"] / cap_alloc["eco_cap_cr"].sum() * 100).round(1)
cap_alloc["ni_per_cap"]= (cap_alloc["net_income_cr"] / cap_alloc["eco_cap_cr"].replace(0, np.nan)).round(3)
 
print("\n  Capital Allocation by Risk Grade:")
print(cap_alloc.to_string(index=False))
cap_alloc.to_csv(os.path.join(OPT_DIR, "capital_allocation_by_grade.csv"), index=False)
 
# ── Optimal capital allocation via constrained optimization ────────────────
grades_list = ["A", "B", "C", "D", "E"]
n_grades = len(grades_list)
 
# Expected net income per ₹ of capital deployed (by grade)
ni_per_rupee = []
el_per_rupee = []
for g in grades_list:
    sub = approved[approved["risk_grade_clean"] == g]
    if len(sub) > 0:
        ni_r  = sub["net_income"].sum() / max(sub["economic_capital"].sum(), 1)
        el_r  = sub["expected_loss"].sum() / max(sub["loan_amount"].sum(), 1)
    else:
        ni_r, el_r = 0.0, 0.15
    ni_per_rupee.append(ni_r)
    el_per_rupee.append(el_r)
 
ni_arr = np.array(ni_per_rupee)
el_arr = np.array(el_per_rupee)
 
# Max capital constraints from risk appetite
max_alloc_pct = np.array([
    PORTFOLIO_TARGETS[g]["max_pct"] for g in grades_list
])
min_alloc_pct = np.array([
    PORTFOLIO_TARGETS[g]["min_pct"] for g in grades_list
])
 
def neg_portfolio_ni(alloc_pct):
    """Objective: maximise net income → minimise negative NI."""
    alloc_pct = np.array(alloc_pct)
    capital   = alloc_pct * TOTAL_CAPITAL_CR * 1e7  # in ₹
    # NI ∝ capital × ni_per_rupee_for_grade
    return -float(np.dot(alloc_pct, ni_arr))
 
def portfolio_el_constraint(alloc_pct):
    """EL must remain below max_el_ceiling (portfolio-level)."""
    portfolio_el = np.dot(alloc_pct, el_arr)
    return RISK_APPETITE["max_npa_ratio"] * 3 - portfolio_el  # must be ≥ 0
 
constraints = [
    {"type": "eq",   "fun": lambda x: np.sum(x) - 1.0},     # must sum to 1
    {"type": "ineq", "fun": portfolio_el_constraint},          # EL ceiling
]
bounds = [(lb, ub) for lb, ub in zip(min_alloc_pct, max_alloc_pct)]
x0 = np.array([PORTFOLIO_TARGETS[g]["target_pct"] for g in grades_list])
x0 = x0 / x0.sum()
 
result = minimize(neg_portfolio_ni, x0, method="SLSQP",
                  bounds=bounds, constraints=constraints,
                  options={"maxiter": 1000, "ftol": 1e-9})
 
optimal_alloc = result.x / result.x.sum() if result.success else x0
optimal_ni    = -result.fun if result.success else 0
 
print("\n  Optimal Capital Allocation (Constrained Optimization):")
current_alloc = np.array([
    approved[approved["risk_grade_clean"] == g].shape[0] / len(approved)
    for g in grades_list
])
 
alloc_df = pd.DataFrame({
    "grade":           grades_list,
    "current_pct":     (current_alloc * 100).round(1),
    "optimal_pct":     (optimal_alloc * 100).round(1),
    "target_pct":      [PORTFOLIO_TARGETS[g]["target_pct"] * 100 for g in grades_list],
    "ni_per_rupee":    ni_arr.round(4),
    "el_per_rupee":    el_arr.round(4),
    "priority":        [PORTFOLIO_TARGETS[g]["priority"] for g in grades_list],
})
alloc_df["rebalance_action"] = alloc_df.apply(
    lambda r: f"↑ Increase +{r['optimal_pct']-r['current_pct']:+.1f}pp"
    if r["optimal_pct"] > r["current_pct"] + 1
    else (f"↓ Reduce {r['optimal_pct']-r['current_pct']:+.1f}pp"
          if r["optimal_pct"] < r["current_pct"] - 1
          else "✅ Hold current"),
    axis=1
)
print(alloc_df.to_string(index=False))
alloc_df.to_csv(os.path.join(OPT_DIR, "optimal_capital_allocation.csv"), index=False)
 
log.info("Capital allocation engine complete. Optimal NI multiplier: %.4f", optimal_ni)
print("\n  ✅  Capital allocation optimization complete.")


══════════════════════════════════════════════════════════════════════
  STEP 4 — CAPITAL ALLOCATION ENGINE
══════════════════════════════════════════════════════════════════════

  Capital Allocation by Risk Grade:
risk_grade_clean  count     aum_cr     ecl_cr  eco_cap_cr  net_income_cr  avg_cap_efficiency  avg_raroc  cap_pct  ni_per_cap
               B    121   3.970832   0.346629    0.346629       0.311744           -1.859598  -1.859598      0.1       0.899
               D     18   0.667284   0.288568    0.288568      -0.091677           -1.212828  -1.212828      0.1      -0.318
               E  24460 802.485477 477.825248  477.825248     -70.193148           -0.611865  -0.611865     99.9      -0.147

  Optimal Capital Allocation (Constrained Optimization):
grade  current_pct  optimal_pct  target_pct  ni_per_rupee  el_per_rupee                                priority    rebalance_action
    A          0.0         16.7        15.0        0.0000        0.1500             High — pr

In [12]:
# =============================================================================
# CELL 8 — STEP 5 & 6: Exposure & Segment Optimization
# =============================================================================
 
section("STEP 5 & 6 — EXPOSURE & SEGMENT PORTFOLIO OPTIMIZATION")
 
# ── Concentration Risk Analysis ────────────────────────────────────────────
def compute_concentration_metrics(df_in, group_col, aum_col="loan_amount"):
    """
    Compute HHI, exposure share, and governance flags for any dimension.
    HHI = Σ(s_i²) where s_i = segment share.
    HHI > 0.25 → high concentration; HHI < 0.10 → diverse.
    """
    if group_col not in df_in.columns:
        return pd.DataFrame()
    total = df_in[aum_col].sum()
    grp = df_in.groupby(group_col)[aum_col].sum().reset_index()
    grp["share_pct"] = (grp[aum_col] / total * 100).round(2)
    grp["hhi_contrib"] = (grp["share_pct"] / 100) ** 2
    hhi = grp["hhi_contrib"].sum()
    grp["concentration_flag"] = grp["share_pct"].apply(
        lambda s: "⚠ HIGH"   if s > 40
                  else "📊 MODERATE" if s > 25
                  else "✅ OK"
    )
    grp["hhi_portfolio"] = round(float(hhi), 4)
    grp["hhi_status"]    = "High" if hhi > 0.25 else "Moderate" if hhi > 0.10 else "Diverse"
    return grp.sort_values(aum_col, ascending=False)
 
 
concentration_dims = {
    "risk_grade_clean":    "Risk Grade",
    "loan_type":           "Loan Product",
    "employment_type":     "Employment Type",
    "acquisition_channel": "Acquisition Channel",
    "urban_semiurban_flag":"Geography (Urban/Semi-Urban)",
}
 
print("\n  Concentration Analysis:")
concentration_reports = {}
for col, label in concentration_dims.items():
    conc = compute_concentration_metrics(approved, col)
    if len(conc):
        concentration_reports[col] = conc
        conc.to_csv(os.path.join(RPT_DIR, f"concentration_{col}.csv"), index=False)
        hhi_val = conc["hhi_portfolio"].iloc[0]
        hhi_status = conc["hhi_status"].iloc[0]
        print(f"\n  {label}: HHI={hhi_val:.4f} ({hhi_status})")
        print(conc[[col, "share_pct", "concentration_flag"]].to_string(index=False))
 
# ── Segment Exposure Optimization ──────────────────────────────────────────
# For each key segment, compute: current exposure vs optimal target
# Optimal target = weight by RAROC × recovery_probability / risk
 
print("\n\n  Segment Exposure Optimization:")
segment_opt_results = {}
 
for seg_col in ["risk_grade_clean", "loan_type", "employment_type"]:
    if seg_col not in approved.columns:
        continue
    seg_perf = approved.groupby(seg_col).agg(
        count          = ("loan_amount",      "count"),
        aum_cr         = ("loan_amount",       lambda x: x.sum()/1e7),
        avg_raroc      = ("raroc",             "mean"),
        avg_pd         = ("pd_score",          "mean"),
        net_income_cr  = ("net_income",        lambda x: x.sum()/1e7),
        avg_rec_prob   = ("recovery_probability","mean"),
    ).reset_index()
 
    total_aum  = seg_perf["aum_cr"].sum()
    seg_perf["current_pct"] = (seg_perf["aum_cr"] / total_aum * 100).round(1)
 
    # Optimal weight ∝ (RAROC × recovery_prob) / max_pd_relative
    score = (
        seg_perf["avg_raroc"].clip(0) * seg_perf["avg_rec_prob"]
        / seg_perf["avg_pd"].clip(lower=0.01)
    )
    score = score.clip(lower=0)
    if score.sum() > 0:
        seg_perf["optimal_pct"] = (score / score.sum() * 100).round(1)
    else:
        seg_perf["optimal_pct"] = seg_perf["current_pct"]
 
    seg_perf["delta_pp"] = (
        seg_perf["optimal_pct"] - seg_perf["current_pct"]
    ).round(1)
    seg_perf["action"] = seg_perf["delta_pp"].apply(
        lambda d: f"↑ GROW +{d:.1f}pp" if d > 2
                  else (f"↓ TRIM {d:.1f}pp" if d < -2
                        else "✅ HOLD")
    )
    segment_opt_results[seg_col] = seg_perf
    seg_perf.to_csv(os.path.join(OPT_DIR, f"segment_optimization_{seg_col}.csv"), index=False)
 
    print(f"\n  Segment Optimization — {seg_col}:")
    print(seg_perf[[seg_col, "current_pct", "optimal_pct",
                    "delta_pp", "action", "avg_raroc"]].to_string(index=False))
 
log.info("Exposure and segment optimization complete.")
print("\n  ✅  Segment exposure optimization complete.")


══════════════════════════════════════════════════════════════════════
  STEP 5 & 6 — EXPOSURE & SEGMENT PORTFOLIO OPTIMIZATION
══════════════════════════════════════════════════════════════════════

  Concentration Analysis:

  Risk Grade: HHI=0.9887 (High)
risk_grade_clean  share_pct concentration_flag
               E      99.43             ⚠ HIGH
               B       0.49               ✅ OK
               D       0.08               ✅ OK

  Loan Product: HHI=0.5023 (High)
          loan_type  share_pct concentration_flag
SME Working Capital      56.94             ⚠ HIGH
      Personal Loan      42.19             ⚠ HIGH
               BNPL       0.87               ✅ OK

  Employment Type: HHI=0.2981 (High)
    employment_type  share_pct concentration_flag
           Salaried      46.87             ⚠ HIGH
      Self-Employed      20.94               ✅ OK
         Gig Worker      11.17               ✅ OK
First-Time Borrower      10.75               ✅ OK
          Sme Owner      10.2

In [13]:
# =============================================================================
# CELL 9 — STEP 7: Risk-Return Optimization (Efficient Frontier)
# =============================================================================
 
section("STEP 7 — RISK-RETURN OPTIMIZATION (EFFICIENT FRONTIER)")
 
# ─────────────────────────────────────────────────────────────────────────
# EFFICIENT FRONTIER FOR LENDING PORTFOLIOS:
# Analogous to Markowitz in asset management, but for credit portfolios.
# X-axis: Portfolio Expected Loss (risk)
# Y-axis: Portfolio Net Income / RAROC (return)
#
# We sweep over the PD approval threshold from strict to aggressive
# and compute risk-return coordinates for each portfolio.
# This reveals the optimal approval threshold that maximises
# return per unit of credit risk taken.
# ─────────────────────────────────────────────────────────────────────────
 
pd_thresholds_ef = np.linspace(0.04, 0.75, 50)
efficient_frontier = []
 
for pd_thresh in pd_thresholds_ef:
    subset = approved[approved["pd_score"] <= pd_thresh]
    if len(subset) < 100:
        continue
 
    aum      = subset["loan_amount"].sum()
    ecl      = subset["expected_loss"].sum()
    ni       = subset["net_income"].sum()
    avg_raroc= subset["raroc"].mean()
    nim      = subset["nim_pct"].mean()
    n_loans  = len(subset)
 
    ecl_pct  = ecl / max(aum, 1) * 100
    ni_cr    = ni / 1e7
    approval = len(subset) / len(approved)
 
    efficient_frontier.append({
        "pd_threshold":   round(pd_thresh, 3),
        "n_loans":        n_loans,
        "approval_rate":  round(approval, 3),
        "ecl_pct_aum":    round(ecl_pct, 3),
        "net_income_cr":  round(ni_cr, 3),
        "avg_raroc":      round(avg_raroc, 4),
        "nim_pct":        round(nim, 3),
        "aum_cr":         round(aum / 1e7, 2),
    })
 
ef_df = pd.DataFrame(efficient_frontier)
 
# Find optimal point on efficient frontier (max Sharpe-like: NI / EL_pct)
ef_df["risk_return_score"] = (
    ef_df["net_income_cr"] / ef_df["ecl_pct_aum"].clip(lower=0.01)
).round(3)
optimal_idx = ef_df["risk_return_score"].idxmax()
optimal_pt  = ef_df.loc[optimal_idx]
 
print(f"\n  Efficient Frontier Analysis:")
print(f"  Data points computed: {len(ef_df)}")
print(f"\n  Optimal Portfolio Point:")
print(f"    PD Threshold    : {optimal_pt['pd_threshold']:.3f}")
print(f"    Approval Rate   : {optimal_pt['approval_rate']:.1%}")
print(f"    Net Income      : ₹{optimal_pt['net_income_cr']:.2f} Cr")
print(f"    ECL/AUM         : {optimal_pt['ecl_pct_aum']:.2f}%")
print(f"    Avg RAROC       : {optimal_pt['avg_raroc']:.3f}")
print(f"    Risk-Return Score: {optimal_pt['risk_return_score']:.3f}")
 
ef_df.to_csv(os.path.join(OPT_DIR, "efficient_frontier_data.csv"), index=False)
 
# ── Multi-dimensional frontier: Grade mix optimization ────────────────────
print("\n  Grade-Mix Efficient Frontier (100 random portfolios):")
 
np.random.seed(SEED)
mixed_frontier = []
 
for trial in range(200):
    # Random grade allocations summing to 1
    alloc = np.random.dirichlet(alpha=[2, 3, 4, 1.5, 0.5])
    alloc = np.clip(alloc, 0.01, 0.50)
    alloc = alloc / alloc.sum()
 
    # Compute portfolio metrics for this mix
    pf_ni   = sum(alloc[i] * grade_risk[grade_risk["risk_grade_clean"]==g]["net_income_cr"].values[0]
                  if g in grade_risk["risk_grade_clean"].values else 0
                  for i, g in enumerate(grades_list))
    pf_ecl  = sum(alloc[i] * grade_risk[grade_risk["risk_grade_clean"]==g]["ecl_cr"].values[0]
                  if g in grade_risk["risk_grade_clean"].values else 0
                  for i, g in enumerate(grades_list))
    pf_raroc= sum(alloc[i] * (grade_risk[grade_risk["risk_grade_clean"]==g]["avg_raroc"].values[0]
                              if g in grade_risk["risk_grade_clean"].values else 0)
                  for i, g in enumerate(grades_list))
 
    mixed_frontier.append({
        "trial":    trial,
        "ni_cr":    round(float(pf_ni), 3),
        "ecl_cr":   round(float(pf_ecl), 3),
        "raroc":    round(float(pf_raroc), 4),
        "grade_A":  round(float(alloc[0]) * 100, 1),
        "grade_B":  round(float(alloc[1]) * 100, 1),
        "grade_C":  round(float(alloc[2]) * 100, 1),
        "grade_D":  round(float(alloc[3]) * 100, 1),
        "grade_E":  round(float(alloc[4]) * 100, 1),
    })
 
mix_ef_df = pd.DataFrame(mixed_frontier)
# Pareto frontier (dominated set removal)
mix_ef_df["is_pareto"] = False
for i, row in mix_ef_df.iterrows():
    dominated = False
    for j, other in mix_ef_df.iterrows():
        if i != j:
            if other["ni_cr"] >= row["ni_cr"] and other["ecl_cr"] <= row["ecl_cr"]:
                dominated = True
                break
    mix_ef_df.loc[i, "is_pareto"] = not dominated
 
pareto_df = mix_ef_df[mix_ef_df["is_pareto"]].sort_values("ni_cr", ascending=False)
print(f"  Pareto-optimal portfolios: {len(pareto_df)}/200")
print(pareto_df[["ni_cr","ecl_cr","raroc","grade_A","grade_B","grade_C"]].head(10).to_string(index=False))
mix_ef_df.to_csv(os.path.join(OPT_DIR, "grade_mix_frontier.csv"), index=False)
pareto_df.to_csv(os.path.join(OPT_DIR, "pareto_optimal_portfolios.csv"), index=False)
 
log.info("Risk-return optimization complete.")
print("\n  ✅  Efficient frontier and Pareto analysis complete.")
 
 


══════════════════════════════════════════════════════════════════════
  STEP 7 — RISK-RETURN OPTIMIZATION (EFFICIENT FRONTIER)
══════════════════════════════════════════════════════════════════════

  Efficient Frontier Analysis:
  Data points computed: 49

  Optimal Portfolio Point:
    PD Threshold    : 0.054
    Approval Rate   : 0.5%
    Net Income      : ₹0.31 Cr
    ECL/AUM         : 8.73%
    Avg RAROC       : -1.860
    Risk-Return Score: 0.036

  Grade-Mix Efficient Frontier (100 random portfolios):
  Pareto-optimal portfolios: 6/200
 ni_cr  ecl_cr   raroc  grade_A  grade_B  grade_C
-0.548   4.918 -0.9753      3.3     48.8     41.8
-0.564   4.912 -0.9127     21.3     44.4     26.6
-0.587   4.868 -0.7227     19.1     35.9     39.9
-0.604   4.839 -0.5685     42.1     29.6     26.3
-0.648   4.833 -0.4863     45.2     18.5     24.1
-0.651   4.801 -0.3647     32.9     15.8     44.9
10:29:40 | INFO     | Risk-return optimization complete.

  ✅  Efficient frontier and Pareto analys

In [14]:
# =============================================================================
# CELL 10 — STEP 8: Approval Strategy Optimization
# =============================================================================
 
section("STEP 8 — APPROVAL STRATEGY OPTIMIZATION")
 
# ── Simulate three strategy scenarios ─────────────────────────────────────
APPROVAL_STRATEGIES = {
    "Aggressive":    {"pd_ceiling": 0.55, "min_raroc": 0.0,
                      "min_income_ratio": 0.0, "max_dti": 10},
    "Balanced":      {"pd_ceiling": 0.30, "min_raroc": 0.5,
                      "min_income_ratio": 0.0, "max_dti": 5},
    "Conservative":  {"pd_ceiling": 0.15, "min_raroc": 1.0,
                      "min_income_ratio": 0.0, "max_dti": 3.5},
    "Prime Only":    {"pd_ceiling": 0.08, "min_raroc": 2.0,
                      "min_income_ratio": 0.0, "max_dti": 2.5},
}
 
strat_results = []
for strat_name, params in APPROVAL_STRATEGIES.items():
    mask = (
        (approved["pd_score"] <= params["pd_ceiling"]) &
        (approved["raroc"]    >= params["min_raroc"])
    )
    subset = approved[mask]
 
    if len(subset) < 10:
        continue
 
    aum   = subset["loan_amount"].sum()
    ecl   = subset["expected_loss"].sum()
    ni    = subset["net_income"].sum()
    gr_int= subset["gross_interest"].sum()
 
    strat_results.append({
        "strategy":       strat_name,
        "n_approved":     len(subset),
        "approval_rate":  round(len(subset) / len(approved) * 100, 1),
        "aum_cr":         round(aum / 1e7, 2),
        "ecl_cr":         round(ecl / 1e7, 2),
        "ecl_to_aum_pct": round(ecl / max(aum, 1) * 100, 2),
        "net_income_cr":  round(ni / 1e7, 2),
        "avg_raroc":      round(subset["raroc"].mean(), 4),
        "avg_pd":         round(subset["pd_score"].mean(), 4),
        "avg_nim":        round(subset["nim_pct"].mean(), 3),
        "pct_profitable": round((subset["net_income"] > 0).mean() * 100, 1),
        "pd_ceiling":     params["pd_ceiling"],
        "min_raroc":      params["min_raroc"],
    })
 
strat_df = pd.DataFrame(strat_results)
print("\n  Approval Strategy Comparison:")
print(strat_df[[
    "strategy", "approval_rate", "ecl_to_aum_pct",
    "net_income_cr", "avg_raroc", "avg_nim", "pct_profitable"
]].to_string(index=False))
strat_df.to_csv(os.path.join(SIM_DIR, "approval_strategy_comparison.csv"), index=False)
 
# ── Optimal threshold by incremental loan analysis ────────────────────────
# Sort by RAROC descending, add loans cumulatively
sorted_loans = approved.sort_values("raroc", ascending=False).reset_index(drop=True)
incremental = []
cum_ni = 0
cum_ecl = 0
cum_aum = 0
 
for i, row in sorted_loans.iterrows():
    cum_ni  += row["net_income"]
    cum_ecl += row["expected_loss"]
    cum_aum += row["loan_amount"]
    if (i + 1) % 500 == 0 or i == len(sorted_loans) - 1:
        incremental.append({
            "n_loans":   i + 1,
            "cum_ni_cr": round(cum_ni / 1e7, 3),
            "cum_ecl_cr":round(cum_ecl / 1e7, 3),
            "ecl_pct":   round(cum_ecl / max(cum_aum, 1) * 100, 3),
            "incr_raroc":round(row["raroc"], 4),
            "n_pct":     round((i + 1) / len(sorted_loans) * 100, 1),
        })
 
incr_df = pd.DataFrame(incremental)
incr_df.to_csv(os.path.join(OPT_DIR, "incremental_loan_analysis.csv"), index=False)
 
# Find point of diminishing returns
ni_peak_idx = incr_df["cum_ni_cr"].idxmax()
optimal_n   = incr_df.loc[ni_peak_idx, "n_loans"]
optimal_pct = incr_df.loc[ni_peak_idx, "n_pct"]
print(f"\n  Optimal loan count for max NI: {optimal_n:,} ({optimal_pct:.1f}% of portfolio)")
print(f"  Peak NI: ₹{incr_df.loc[ni_peak_idx, 'cum_ni_cr']:.2f} Cr")
 
log.info("Approval strategy optimization complete.")
print("\n  ✅  Approval strategy optimization complete.")


══════════════════════════════════════════════════════════════════════
  STEP 8 — APPROVAL STRATEGY OPTIMIZATION
══════════════════════════════════════════════════════════════════════

  Approval Strategy Comparison:
    strategy  approval_rate  ecl_to_aum_pct  net_income_cr  avg_raroc  avg_nim  pct_profitable
  Aggressive            5.4           65.41          14.99     0.2613    2.872           100.0
    Balanced            0.3            9.07           0.30     0.9150    2.860           100.0
Conservative            0.1            9.93           0.14     1.2321    3.616           100.0

  Optimal loan count for max NI: 4,000 (16.3% of portfolio)
  Peak NI: ₹46.26 Cr
10:29:56 | INFO     | Approval strategy optimization complete.

  ✅  Approval strategy optimization complete.


In [15]:
# =============================================================================
# CELL 11 — STEP 9 & 10: Pricing & Collections Integration
# =============================================================================
 
section("STEP 9 & 10 — PRICING & COLLECTIONS PORTFOLIO INTEGRATION")
 
# ── Pricing adequacy analysis ──────────────────────────────────────────────
# Is current pricing sufficient to cover risk at portfolio level?
 
approved["pricing_adequacy_gap"] = (
    approved["gross_interest"]
    - approved["expected_loss"]
    - 500 * (approved["loan_tenure_months"].fillna(24) / 12)  # servicing
).round(2)
 
approved["pricing_adequate"] = (approved["pricing_adequacy_gap"] > 0).astype(int)
print(f"\n  Pricing Adequacy:")
print(f"  % Loans adequately priced: {approved['pricing_adequate'].mean()*100:.1f}%")
print(f"  Total pricing gap (negative=under-priced): ₹{approved['pricing_adequacy_gap'].sum()/1e7:.2f} Cr")
 
by_grade_pricing = approved.groupby("risk_grade_clean").agg(
    pct_adequate   = ("pricing_adequate",    "mean"),
    avg_gap        = ("pricing_adequacy_gap","mean"),
    total_gap_cr   = ("pricing_adequacy_gap",lambda x: x.sum()/1e7),
).reset_index()
by_grade_pricing["pct_adequate"] = (by_grade_pricing["pct_adequate"] * 100).round(1)
print(by_grade_pricing.to_string(index=False))
by_grade_pricing.to_csv(os.path.join(RPT_DIR, "pricing_adequacy_by_grade.csv"), index=False)
 
# ── Pricing uplift optimization ────────────────────────────────────────────
# For under-priced grades, compute minimum rate increase needed
under_priced = approved[approved["pricing_adequate"] == 0].copy()
if len(under_priced) > 0:
    under_priced["min_rate_increase_needed"] = (
        under_priced["expected_loss"]
        / under_priced["loan_amount"].clip(lower=1)
        / (under_priced["loan_tenure_months"].fillna(24) / 12).clip(lower=0.25)
        * 100
        - under_priced["interest_rate"] + 12.5  # base + ops
    ).clip(lower=0).round(2)
 
    uplift_by_grade = under_priced.groupby("risk_grade_clean").agg(
        count                  = ("loan_amount", "count"),
        avg_rate_increase_needed=("min_rate_increase_needed","mean"),
        total_revenue_uplift_cr = ("min_rate_increase_needed",
                                   lambda x: (x * under_priced.loc[x.index, "loan_amount"] / 100).sum()/1e7),
    ).reset_index()
    print("\n  Pricing Uplift Recommendations:")
    print(uplift_by_grade.to_string(index=False))
 
# ── Collections impact on portfolio profitability ──────────────────────────
# Recovery-adjusted profitability
approved["collections_adj_profit"] = (
    approved["net_income"]
    + approved["expected_loss"] * approved["recovery_probability"]
    - 300 * (approved["delinquency_risk_prob"] > 0.40).astype(float)  # collections cost
).round(2)
 
collections_uplift = (
    approved["collections_adj_profit"].sum() - approved["net_income"].sum()
) / 1e7
 
print(f"\n  Collections Impact on Portfolio:")
print(f"  Recovery-adjusted NI uplift: ₹{collections_uplift:.2f} Cr")
print(f"  Avg cure rate proxy: {approved['recovery_probability'].mean():.1%}")
 
# Collections ROI by segment
coll_seg = approved.groupby("risk_grade_clean").agg(
    avg_recovery_prob  = ("recovery_probability",    "mean"),
    ecl_cr             = ("expected_loss",            lambda x: x.sum()/1e7),
    recoverable_cr     = ("expected_loss",
                          lambda x: (x * approved.loc[x.index, "recovery_probability"]).sum()/1e7),
    collections_adj_cr = ("collections_adj_profit",  lambda x: x.sum()/1e7),
    base_ni_cr         = ("net_income",               lambda x: x.sum()/1e7),
).reset_index()
coll_seg["recovery_uplift_cr"] = (
    coll_seg["collections_adj_cr"] - coll_seg["base_ni_cr"]
).round(3)
 
print("\n  Collections ROI by Grade:")
print(coll_seg.to_string(index=False))
coll_seg.to_csv(os.path.join(RPT_DIR, "collections_portfolio_impact.csv"), index=False)
 
log.info("Pricing and collections integration complete.")
print("\n  ✅  Pricing-portfolio and collections-portfolio integration complete.")


══════════════════════════════════════════════════════════════════════
  STEP 9 & 10 — PRICING & COLLECTIONS PORTFOLIO INTEGRATION
══════════════════════════════════════════════════════════════════════

  Pricing Adequacy:
  % Loans adequately priced: 16.3%
  Total pricing gap (negative=under-priced): ₹-67.51 Cr
risk_grade_clean  pct_adequate       avg_gap  total_gap_cr
               B          96.7  26763.997355      0.323844
               D           5.6 -49931.688333     -0.089877
               E          15.9 -27697.117126    -67.747148

  Pricing Uplift Recommendations:
risk_grade_clean  count  avg_rate_increase_needed  total_revenue_uplift_cr
               B      4                  3.845000                 0.000193
               D     17                 13.200000                 0.069804
               E  20562                 16.619614                75.346168

  Collections Impact on Portfolio:
  Recovery-adjusted NI uplift: ₹176.39 Cr
  Avg cure rate proxy: 37.0%

  Coll

In [16]:
# =============================================================================
# CELL 12 — STEP 11: Scenario Simulation Engine
# =============================================================================
 
section("STEP 11 — SCENARIO SIMULATION ENGINE")
 
PORTFOLIO_SCENARIOS = {
    "Baseline": {
        "pd_shock":     0.00, "income_shock": 0.00,
        "cure_shock":   0.00, "rate_shock":   0.00,
        "vol_shock":    0.00,
        "description":  "Current conditions. No macro changes.",
    },
    "Mild Recession": {
        "pd_shock":     0.04, "income_shock": -0.05,
        "cure_shock":  -0.05, "rate_shock":    0.00,
        "vol_shock":    0.10,
        "description":  "Mild GDP contraction. Slight income drop.",
    },
    "Moderate Recession": {
        "pd_shock":     0.08, "income_shock": -0.12,
        "cure_shock":  -0.10, "rate_shock":    0.00,
        "vol_shock":    0.20,
        "description":  "Moderate recession. Rising unemployment.",
    },
    "Severe Recession": {
        "pd_shock":     0.15, "income_shock": -0.20,
        "cure_shock":  -0.20, "rate_shock":    0.00,
        "vol_shock":    0.35,
        "description":  "Severe downturn. NPA spike scenario.",
    },
    "Rate Cap Tightening": {
        "pd_shock":     0.00, "income_shock":  0.00,
        "cure_shock":   0.00, "rate_shock":   -5.00,
        "vol_shock":    0.00,
        "description":  "RBI rate cap reduced by 5pp.",
    },
    "Aggressive Growth": {
        "pd_shock":     0.05, "income_shock":  0.00,
        "cure_shock":   0.00, "rate_shock":   -2.00,
        "vol_shock":    0.05,
        "description":  "Approval rate increase 20pp. Lower quality intake.",
    },
    "Collections Breakdown": {
        "pd_shock":     0.02, "income_shock":  0.00,
        "cure_shock":  -0.30, "rate_shock":    0.00,
        "vol_shock":    0.00,
        "description":  "Collections effectiveness drops 30%. Higher write-off.",
    },
    "Fintech Credit Crunch": {
        "pd_shock":     0.10, "income_shock": -0.15,
        "cure_shock":  -0.15, "rate_shock":    2.00,
        "vol_shock":    0.25,
        "description":  "Liquidity crunch. Rising cost of funds.",
    },
}
 
scenario_results = []
for scen_name, params in PORTFOLIO_SCENARIOS.items():
    s = approved.copy()
 
    # Apply shocks
    s["stressed_pd"]   = np.clip(s["pd_score"] + params["pd_shock"], 0.001, 0.999)
    s["stressed_rate"] = np.clip(s["interest_rate"] + params["rate_shock"], 8.0, 40.0)
    s["stressed_rec"]  = np.clip(s["recovery_probability"] + params["cure_shock"], 0.01, 0.95)
 
    # Recompute EL and profitability
    tenure_yr = s["loan_tenure_months"].fillna(24) / 12
    stressed_cum_pd = 1 - (1 - s["stressed_pd"]) ** tenure_yr
    s["stressed_el"]  = (stressed_cum_pd * s["lgd"] * s["loan_amount"]).round(2)
 
    mr_s  = s["stressed_rate"] / 100 / 12
    n_s   = s["loan_tenure_months"].fillna(24).astype(int)
    emi_s = s["loan_amount"] * mr_s * (1 + mr_s)**n_s / ((1 + mr_s)**n_s - 1)
    s["stressed_gross_int"] = (emi_s * n_s - s["loan_amount"]).round(2)
    s["stressed_ni"]        = (s["stressed_gross_int"] - s["stressed_el"] - 500 * tenure_yr - 1000).round(2)
    s["stressed_rec_adj_ni"]= (s["stressed_ni"] + s["stressed_el"] * s["stressed_rec"]).round(2)
    s["stressed_raroc"]     = (s["stressed_ni"] / (s["stressed_el"] * 12.5 * 0.08).clip(lower=1)).round(4)
 
    aum = s["loan_amount"].sum()
    scenario_results.append({
        "scenario":              scen_name,
        "avg_stressed_pd":       round(s["stressed_pd"].mean(), 4),
        "total_ecl_cr":          round(s["stressed_el"].sum() / 1e7, 2),
        "ecl_to_aum_pct":        round(s["stressed_el"].sum() / max(aum, 1) * 100, 2),
        "net_income_cr":         round(s["stressed_ni"].sum() / 1e7, 2),
        "rec_adj_ni_cr":         round(s["stressed_rec_adj_ni"].sum() / 1e7, 2),
        "avg_raroc":             round(s["stressed_raroc"].mean(), 4),
        "pct_profitable":        round((s["stressed_ni"] > 0).mean() * 100, 1),
        "avg_recovery_prob":     round(s["stressed_rec"].mean(), 4),
        "ni_delta_vs_baseline":  0.0,
        "ecl_delta_vs_baseline": 0.0,
        "description":           params["description"],
    })
 
scen_df = pd.DataFrame(scenario_results)
base_ni  = scen_df.loc[scen_df["scenario"] == "Baseline", "net_income_cr"].values[0]
base_ecl = scen_df.loc[scen_df["scenario"] == "Baseline", "total_ecl_cr"].values[0]
scen_df["ni_delta_vs_baseline"]  = (scen_df["net_income_cr"]  - base_ni).round(2)
scen_df["ecl_delta_vs_baseline"] = (scen_df["total_ecl_cr"] - base_ecl).round(2)
 
print("\n  Portfolio Scenario Simulation Results:")
print(scen_df[[
    "scenario", "avg_stressed_pd", "ecl_to_aum_pct",
    "net_income_cr", "avg_raroc", "ni_delta_vs_baseline"
]].to_string(index=False))
scen_df.to_csv(os.path.join(SIM_DIR, "portfolio_scenario_results.csv"), index=False)
 
log.info("Scenario simulation engine complete.")
print("\n  ✅  Scenario simulation complete. 8 scenarios analyzed.")


══════════════════════════════════════════════════════════════════════
  STEP 11 — SCENARIO SIMULATION ENGINE
══════════════════════════════════════════════════════════════════════

  Portfolio Scenario Simulation Results:
             scenario  avg_stressed_pd  ecl_to_aum_pct  net_income_cr  avg_raroc  ni_delta_vs_baseline
             Baseline           0.5593           59.28         -69.97    -0.6184                  0.00
       Mild Recession           0.5993           60.82         -82.40    -0.6105                -12.43
   Moderate Recession           0.6393           62.17         -93.34    -0.6035                -23.37
     Severe Recession           0.7093           64.18        -109.50    -0.5917                -39.53
  Rate Cap Tightening           0.5593           59.28        -160.41    -0.7634                -90.44
    Aggressive Growth           0.6093           61.17        -121.99    -0.6649                -52.02
Collections Breakdown           0.5793           60.07 

In [17]:
# =============================================================================
# CELL 13 — STEP 12: Stress Testing Framework
# =============================================================================
 
section("STEP 12 — STRESS TESTING FRAMEWORK")
 
# ── Enterprise stress testing ────────────────────────────────────────────
# Factor-based stress: each variable stressed independently and in combination
 
STRESS_FACTORS = {
    "PD Shock":          {"pd_mult": [1.0, 1.2, 1.5, 2.0, 3.0]},
    "Income Drop":       {"income_mult": [1.0, 0.95, 0.85, 0.75, 0.60]},
    "Collections Cure":  {"cure_mult": [1.0, 0.90, 0.75, 0.60, 0.40]},
    "Rate Cap":          {"rate_adj": [0, -2, -4, -6, -8]},
}
 
stress_sweep_results = []
for factor_name, factor_vals in STRESS_FACTORS.items():
    factor_key = list(factor_vals.keys())[0]
    for val in list(factor_vals.values())[0]:
        s = approved.copy()
        tenure_yr = s["loan_tenure_months"].fillna(24) / 12
 
        if factor_key == "pd_mult":
            s["stressed_pd"] = np.clip(s["pd_score"] * val, 0.001, 0.999)
        elif factor_key == "income_mult":
            s["stressed_pd"] = np.clip(s["pd_score"] / val, 0.001, 0.999)
        elif factor_key == "cure_mult":
            s["stressed_pd"] = s["pd_score"].copy()
        else:
            s["stressed_pd"] = s["pd_score"].copy()
 
        if factor_key == "rate_adj":
            s["stressed_rate"] = np.clip(s["interest_rate"] + val, 8, 40)
        else:
            s["stressed_rate"] = s["interest_rate"].copy()
 
        if factor_key == "cure_mult":
            s["stressed_rec"] = np.clip(s["recovery_probability"] * val, 0.01, 0.95)
        else:
            s["stressed_rec"] = s["recovery_probability"].copy()
 
        cum_pd_s = 1 - (1 - s["stressed_pd"]) ** tenure_yr
        s["stressed_el"] = cum_pd_s * s["lgd"] * s["loan_amount"]
 
        mr_s = s["stressed_rate"] / 100 / 12
        n_s  = s["loan_tenure_months"].fillna(24).astype(int)
        emi_s= s["loan_amount"] * mr_s * (1+mr_s)**n_s / ((1+mr_s)**n_s - 1)
        s["stressed_gi"] = emi_s * n_s - s["loan_amount"]
        s["stressed_ni"] = s["stressed_gi"] - s["stressed_el"] - 500 * tenure_yr - 1000
 
        aum = s["loan_amount"].sum()
        stress_sweep_results.append({
            "stress_factor":     factor_name,
            "factor_value":      val,
            "total_ecl_cr":      round(s["stressed_el"].sum() / 1e7, 2),
            "ecl_to_aum_pct":    round(s["stressed_el"].sum() / max(aum, 1) * 100, 2),
            "net_income_cr":     round(s["stressed_ni"].sum() / 1e7, 2),
            "pct_profitable":    round((s["stressed_ni"] > 0).mean() * 100, 1),
            "pct_npa_breach":    round((s["stressed_pd"] > 0.30).mean() * 100, 1),
        })
 
stress_sweep_df = pd.DataFrame(stress_sweep_results)
stress_sweep_df.to_csv(os.path.join(STR_DIR, "stress_sweep_results.csv"), index=False)
 
# Resilience score: how much PD shock can the portfolio absorb before NI < 0?
pd_stress = stress_sweep_df[stress_sweep_df["stress_factor"] == "PD Shock"]
breaking_pd_mult = None
for _, row in pd_stress.iterrows():
    if row["net_income_cr"] < 0:
        breaking_pd_mult = row["factor_value"]
        break
 
if breaking_pd_mult:
    print(f"\n  Portfolio breaks at PD multiplier: {breaking_pd_mult}×")
    print(f"  Current avg PD: {approved['pd_score'].mean():.3%}")
    print(f"  Breaking-point PD: {approved['pd_score'].mean() * breaking_pd_mult:.3%}")
    resilience_score = (breaking_pd_mult - 1) * 100
    print(f"  Resilience score: {resilience_score:.0f}/200 (100 = withstands 2× PD shock)")
else:
    resilience_score = 200
    print(f"  Portfolio is resilient to all stress scenarios tested.")
 
with open(os.path.join(STR_DIR, "stress_test_summary.json"), "w") as f:
    json.dump({
        "breaking_pd_multiplier": breaking_pd_mult,
        "resilience_score":       resilience_score,
        "scenarios_tested":       len(PORTFOLIO_SCENARIOS),
        "stress_factors":         list(STRESS_FACTORS.keys()),
    }, f, indent=2)
 
# Heatmap data: scenario × grade NI impact
stress_heatmap_data = {}
for scen_name, params in list(PORTFOLIO_SCENARIOS.items())[:6]:
    s = approved.copy()
    s["stressed_pd"] = np.clip(s["pd_score"] + params["pd_shock"], 0.001, 0.999)
    s["stressed_rate"]= np.clip(s["interest_rate"] + params["rate_shock"], 8, 40)
    tenure_yr = s["loan_tenure_months"].fillna(24) / 12
    cum_pd_s = 1 - (1 - s["stressed_pd"]) ** tenure_yr
    s["stressed_el"]  = cum_pd_s * s["lgd"] * s["loan_amount"]
    mr_s = s["stressed_rate"] / 100 / 12
    n_s  = s["loan_tenure_months"].fillna(24).astype(int)
    emi_s= s["loan_amount"] * mr_s * (1+mr_s)**n_s / ((1+mr_s)**n_s - 1)
    s["stressed_gi"] = emi_s * n_s - s["loan_amount"]
    s["stressed_ni"] = s["stressed_gi"] - s["stressed_el"] - 500 * tenure_yr - 1000
 
    for g in grades_list:
        sub = s[s["risk_grade_clean"] == g]
        ni_g = sub["stressed_ni"].sum() / 1e7 if len(sub) > 0 else 0
        stress_heatmap_data.setdefault(scen_name, {})[g] = round(float(ni_g), 2)
 
stress_heatmap_df = pd.DataFrame(stress_heatmap_data).T
stress_heatmap_df.to_csv(os.path.join(STR_DIR, "stress_heatmap_ni_by_grade.csv"))
 
log.info("Stress testing framework complete. Resilience score: %s", resilience_score)
print(f"\n  ✅  Stress testing complete. Resilience score: {resilience_score:.0f}/200")
 


══════════════════════════════════════════════════════════════════════
  STEP 12 — STRESS TESTING FRAMEWORK
══════════════════════════════════════════════════════════════════════

  Portfolio breaks at PD multiplier: 1.0×
  Current avg PD: 55.935%
  Breaking-point PD: 55.935%
  Resilience score: 0/200 (100 = withstands 2× PD shock)
10:30:46 | INFO     | Stress testing framework complete. Resilience score: 0.0

  ✅  Stress testing complete. Resilience score: 0/200


In [18]:
# =============================================================================
# CELL 14 — STEP 13: Concentration Risk Management
# =============================================================================
 
section("STEP 13 — CONCENTRATION RISK MANAGEMENT")
 
# ── Concentration governance framework ────────────────────────────────────
print("\n  Concentration Governance Framework:")
 
concentration_governance = {}
CONC_LIMITS = {
    "risk_grade_clean":    {"max_single_share": 0.40, "name": "Risk Grade"},
    "loan_type":           {"max_single_share": 0.50, "name": "Loan Product"},
    "employment_type":     {"max_single_share": 0.40, "name": "Employment Type"},
    "acquisition_channel": {"max_single_share": 0.45, "name": "Acquisition Channel"},
    "urban_semiurban_flag":{"max_single_share": 0.60, "name": "Geography"},
}
 
all_breaches = []
for col, limits in CONC_LIMITS.items():
    if col not in approved.columns:
        continue
    shares = (approved.groupby(col)["loan_amount"].sum()
               / approved["loan_amount"].sum())
    breaches = shares[shares > limits["max_single_share"]]
    print(f"\n  {limits['name']}:")
    for seg, share in shares.sort_values(ascending=False).items():
        flag = "❌ BREACH" if share > limits["max_single_share"] else \
               "⚠ WATCH"   if share > limits["max_single_share"] * 0.80 else "✅ OK"
        print(f"    {seg:<35}: {share:.1%} {flag}")
        if share > limits["max_single_share"]:
            all_breaches.append({
                "dimension":  limits["name"],
                "segment":    seg,
                "share_pct":  round(share * 100, 1),
                "limit_pct":  limits["max_single_share"] * 100,
                "excess_pp":  round((share - limits["max_single_share"]) * 100, 1),
                "action":     f"Reduce {seg} by {round((share - limits['max_single_share']*0.9)*100, 1)}pp",
            })
    concentration_governance[col] = {
        "max_observed_share": float(shares.max()),
        "limit":              limits["max_single_share"],
        "breach":             bool(len(breaches) > 0),
        "breach_segments":    list(breaches.index),
    }
 
breach_df = pd.DataFrame(all_breaches)
if len(breach_df):
    print(f"\n  ⚠  {len(breach_df)} concentration breaches detected:")
    print(breach_df.to_string(index=False))
else:
    print("\n  ✅  All concentration limits within governance thresholds.")
 
breach_df.to_csv(os.path.join(GOV_DIR, "concentration_breaches.csv"), index=False)
 
with open(os.path.join(GOV_DIR, "concentration_governance_report.json"), "w") as f:
    json.dump(concentration_governance, f, indent=2)
 
# ── Correlated delinquency cluster detection ──────────────────────────────
# Segments that tend to default together represent correlated risk
corr_matrix = approved.pivot_table(
    values="default_flag",
    index=approved.index // max(1, len(approved) // 100),  # 100 cohorts
    columns="risk_grade_clean",
    aggfunc="mean"
).fillna(0)
if corr_matrix.shape[1] > 1:
    grade_corr = corr_matrix.corr().round(3)
    grade_corr.to_csv(os.path.join(RPT_DIR, "grade_default_correlation.csv"))
    print("\n  Grade Default Correlation (identifies correlated risk clusters):")
    print(grade_corr.to_string())
 
log.info("Concentration risk management complete.")
print("\n  ✅  Concentration risk management complete.")


══════════════════════════════════════════════════════════════════════
  STEP 13 — CONCENTRATION RISK MANAGEMENT
══════════════════════════════════════════════════════════════════════

  Concentration Governance Framework:

  Risk Grade:
    E                                  : 99.4% ❌ BREACH
    B                                  : 0.5% ✅ OK
    D                                  : 0.1% ✅ OK

  Loan Product:
    SME Working Capital                : 56.9% ❌ BREACH
    Personal Loan                      : 42.2% ⚠ WATCH
    BNPL                               : 0.9% ✅ OK

  Employment Type:
    Salaried                           : 46.9% ❌ BREACH
    Self-Employed                      : 20.9% ✅ OK
    Gig Worker                         : 11.2% ✅ OK
    First-Time Borrower                : 10.7% ✅ OK
    Sme Owner                          : 10.3% ✅ OK

  Acquisition Channel:
    Organic Search                     : 25.9% ✅ OK
    Referral                           : 20.1% ✅ OK
    App Stor

In [19]:
# =============================================================================
# CELL 15 — STEP 14: Dynamic Portfolio Rebalancing
# =============================================================================
 
section("STEP 14 — DYNAMIC PORTFOLIO REBALANCING")
 
# ─────────────────────────────────────────────────────────────────────────
# REBALANCING LOGIC:
# Compare current portfolio composition vs optimal allocation.
# For each dimension (grade, product, segment), compute:
#   - Current weight vs target weight
#   - Rebalancing action (grow, trim, hold)
#   - Revenue impact of rebalancing
#   - Risk impact of rebalancing
# Constraints:
#   - Cannot instantly reduce a segment (existing loans have tenure)
#   - New originations can be steered toward target mix
#   - Collections can improve risk profile of existing book
# ─────────────────────────────────────────────────────────────────────────
 
def compute_rebalancing_plan(df_in, target_alloc_df):
    """
    Generate a dynamic rebalancing plan.
    target_alloc_df: DataFrame with 'grade' and 'optimal_pct' columns.
    """
    current = df_in.groupby("risk_grade_clean").agg(
        n_loans    = ("loan_amount", "count"),
        aum_cr     = ("loan_amount", lambda x: x.sum()/1e7),
        avg_raroc  = ("raroc",       "mean"),
        avg_pd     = ("pd_score",    "mean"),
    ).reset_index()
    current["current_pct"] = (current["aum_cr"] / current["aum_cr"].sum() * 100).round(1)
 
    plan = current.merge(
        target_alloc_df[["grade", "optimal_pct", "priority"]],
        left_on="risk_grade_clean", right_on="grade", how="left"
    )
    plan["delta_pp"]     = (plan["optimal_pct"] - plan["current_pct"]).round(1)
    plan["delta_aum_cr"] = (plan["delta_pp"] / 100 * plan["aum_cr"].sum()).round(2)
 
    def action_label(delta, avg_raroc):
        if delta > 3:
            return f"↑ GROW: +{delta:.1f}pp — RAROC={avg_raroc:.2f} supports expansion"
        elif delta > 1:
            return f"↑ TILT: +{delta:.1f}pp — modestly increase originations"
        elif delta < -3:
            return f"↓ TRIM: {delta:.1f}pp — redirect new originations"
        elif delta < -1:
            return f"↓ REDUCE: {delta:.1f}pp — slow new disbursements"
        else:
            return "✅ HOLD: within ±1pp of target"
 
    plan["rebalancing_action"] = plan.apply(
        lambda r: action_label(r["delta_pp"], r["avg_raroc"]), axis=1
    )
 
    # Revenue impact of rebalancing
    plan["ni_per_cr_aum"] = (
        df_in.groupby("risk_grade_clean")["net_income"].sum() /
        df_in.groupby("risk_grade_clean")["loan_amount"].sum().clip(lower=1)
    ).reindex(plan["risk_grade_clean"]).values
 
    plan["expected_ni_uplift_cr"] = (
        plan["delta_aum_cr"] * plan["ni_per_cr_aum"].fillna(0)
    ).round(3)
 
    total_uplift = plan["expected_ni_uplift_cr"].sum()
    plan["pct_contribution_to_uplift"] = (
        plan["expected_ni_uplift_cr"] / max(abs(total_uplift), 0.001) * 100
    ).round(1)
 
    return plan, total_uplift
 
 
rebal_plan, total_ni_uplift = compute_rebalancing_plan(approved, alloc_df)
 
print("\n  Dynamic Rebalancing Plan:")
print(rebal_plan[[
    "risk_grade_clean", "current_pct", "optimal_pct",
    "delta_pp", "expected_ni_uplift_cr", "rebalancing_action"
]].to_string(index=False))
print(f"\n  Total NI uplift from rebalancing: ₹{total_ni_uplift:.2f} Cr")
 
rebal_plan.to_csv(os.path.join(OPT_DIR, "portfolio_rebalancing_plan.csv"), index=False)
 
# ── Monthly rebalancing simulation (24 months) ────────────────────────────
monthly_rebalance = []
current_mix = {g: float(v) for g, v in
               (approved.groupby("risk_grade_clean").size() / len(approved)).items()}
target_mix  = {row["grade"]: row["optimal_pct"]/100 for _, row in alloc_df.iterrows()}
 
for month in range(1, 25):
    # Gradual convergence: 5% of gap closed per month
    for g in grades_list:
        curr = current_mix.get(g, 0)
        tgt  = target_mix.get(g, 0)
        current_mix[g] = curr + (tgt - curr) * 0.05
 
    # Compute implied portfolio NI at current mix
    implied_ni = sum(
        current_mix.get(g, 0) *
        grade_risk[grade_risk["risk_grade_clean"]==g]["net_income_cr"].values[0]
        if g in grade_risk["risk_grade_clean"].values else 0
        for g in grades_list
    )
    monthly_rebalance.append({
        "month": month,
        "implied_ni_cr": round(float(implied_ni), 3),
        "mix_A": round(current_mix.get("A", 0) * 100, 1),
        "mix_B": round(current_mix.get("B", 0) * 100, 1),
        "mix_C": round(current_mix.get("C", 0) * 100, 1),
        "mix_D": round(current_mix.get("D", 0) * 100, 1),
        "mix_E": round(current_mix.get("E", 0) * 100, 1),
    })
 
monthly_rebal_df = pd.DataFrame(monthly_rebalance)
monthly_rebal_df.to_csv(os.path.join(SIM_DIR, "monthly_rebalance_simulation.csv"), index=False)
log.info("Dynamic portfolio rebalancing complete.")
print("\n  ✅  Portfolio rebalancing plan and 24-month simulation complete.")


══════════════════════════════════════════════════════════════════════
  STEP 14 — DYNAMIC PORTFOLIO REBALANCING
══════════════════════════════════════════════════════════════════════

  Dynamic Rebalancing Plan:
risk_grade_clean  current_pct  optimal_pct  delta_pp  expected_ni_uplift_cr                               rebalancing_action
               B          0.5         27.8      27.3                 17.299 ↑ GROW: +27.3pp — RAROC=-1.86 supports expansion
               D          0.1         11.1      11.0                -12.197 ↑ GROW: +11.0pp — RAROC=-1.21 supports expansion
               E         99.4          5.6     -93.8                 66.222      ↓ TRIM: -93.8pp — redirect new originations

  Total NI uplift from rebalancing: ₹71.32 Cr
10:31:21 | INFO     | Dynamic portfolio rebalancing complete.

  ✅  Portfolio rebalancing plan and 24-month simulation complete.


In [20]:
# =============================================================================
# CELL 16 — STEP 15 & 16: Executive Portfolio Dashboards & Advanced Visuals
# =============================================================================
 
section("STEP 15 & 16 — EXECUTIVE DASHBOARDS & ADVANCED VISUAL ANALYTICS")
 
# ── Figure 1: Portfolio Health Overview ───────────────────────────────────
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.40)
fig.suptitle(
    "Executive Portfolio Intelligence Dashboard\n"
    "Digital Lending Portfolio Optimization — Module 9",
    fontsize=16, fontweight="bold", color=PAL["primary"]
)
 
# P1: Grade distribution (pie)
ax1 = fig.add_subplot(gs[0, 0])
grade_counts = approved["risk_grade_clean"].value_counts()
gc_colors    = [GRADE_COLORS.get(g, PAL["neutral"]) for g in grade_counts.index]
ax1.pie(
    grade_counts.values,
    labels=[f"Grade {g}\n{v:,}" for g, v in grade_counts.items()],
    colors=gc_colors, autopct="%1.1f%%", startangle=90, pctdistance=0.75
)
ax1.set_title("Portfolio Grade Distribution")
 
# P2: RAROC by grade
ax2 = fig.add_subplot(gs[0, 1])
raroc_by_grade = approved.groupby("risk_grade_clean")["raroc"].mean().reindex(grades_list)
rc = [PAL["success"] if v > 1 else PAL["warning"] if v > 0 else PAL["danger"]
       for v in raroc_by_grade.fillna(0)]
bars = ax2.bar(raroc_by_grade.index, raroc_by_grade.values, color=rc)
ax2.axhline(1.0, color=PAL["primary"], linestyle="--", lw=1.5, label="Target RAROC=1.0")
ax2.axhline(0.0, color="black", lw=0.8)
ax2.set_title("Avg RAROC by Risk Grade")
ax2.set_ylabel("RAROC"); ax2.legend(fontsize=8)
for bar, v in zip(bars, raroc_by_grade.fillna(0)):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.02, f"{v:.2f}", ha="center", fontsize=9)
 
# P3: ECL decomposition
ax3 = fig.add_subplot(gs[0, 2])
ecl_by_grade = approved.groupby("risk_grade_clean")["expected_loss"].sum().reindex(grades_list) / 1e7
ni_by_grade  = approved.groupby("risk_grade_clean")["net_income"].sum().reindex(grades_list) / 1e7
x_pos = np.arange(len(grades_list))
w = 0.35
ax3.bar(x_pos - w/2, ecl_by_grade.fillna(0), w, color=PAL["danger"],   label="ECL (₹Cr)")
ax3.bar(x_pos + w/2, ni_by_grade.fillna(0),  w, color=PAL["success"],  label="NI (₹Cr)")
ax3.set_xticks(x_pos); ax3.set_xticklabels(grades_list)
ax3.axhline(0, color="black", lw=0.8)
ax3.set_title("ECL vs Net Income by Grade (₹ Cr)")
ax3.set_ylabel("₹ Crore"); ax3.legend(fontsize=8)
 
# P4: Efficient frontier
ax4 = fig.add_subplot(gs[0, 3])
ef_scatter = ef_df.copy()
sc = ax4.scatter(
    ef_scatter["ecl_pct_aum"],
    ef_scatter["net_income_cr"],
    c=ef_scatter["avg_raroc"],
    cmap="RdYlGn", s=30, alpha=0.7
)
plt.colorbar(sc, ax=ax4, label="Avg RAROC")
ax4.scatter(
    optimal_pt["ecl_pct_aum"], optimal_pt["net_income_cr"],
    c="black", s=200, zorder=5, marker="*", label="Optimal point"
)
ax4.set_title("Lending Efficient Frontier")
ax4.set_xlabel("ECL/AUM (%)"); ax4.set_ylabel("Net Income (₹ Cr)")
ax4.legend(fontsize=8)
 
# P5: Scenario impact bar chart
ax5 = fig.add_subplot(gs[1, :2])
scen_names  = scen_df["scenario"].tolist()
ni_deltas   = scen_df["ni_delta_vs_baseline"].tolist()
ecl_deltas  = scen_df["ecl_delta_vs_baseline"].tolist()
x_s = np.arange(len(scen_names))
w_s = 0.35
ni_c  = [PAL["success"] if v >= 0 else PAL["danger"] for v in ni_deltas]
ecl_c = [PAL["danger"]  if v >= 0 else PAL["success"] for v in ecl_deltas]
ax5.bar(x_s - w_s/2, ni_deltas,  w_s, color=ni_c,  label="NI Δ vs Baseline (₹Cr)")
ax5.bar(x_s + w_s/2, ecl_deltas, w_s, color=ecl_c, alpha=0.75, label="ECL Δ vs Baseline (₹Cr)")
ax5.axhline(0, color="black", lw=0.8)
ax5.set_xticks(x_s)
ax5.set_xticklabels([s.replace(" ", "\n") for s in scen_names], fontsize=7)
ax5.set_title("Scenario Impact — NI & ECL Delta vs Baseline")
ax5.set_ylabel("₹ Crore"); ax5.legend(fontsize=8)
 
# P6: Concentration risk matrix
ax6 = fig.add_subplot(gs[1, 2:])
if "risk_grade_clean" in concentration_reports and "loan_type" in concentration_reports:
    conc_pivot = approved.pivot_table(
        values="loan_amount",
        index="risk_grade_clean",
        columns="loan_type",
        aggfunc="sum",
        fill_value=0
    ) / 1e7
    cmap_conc = LinearSegmentedColormap.from_list("conc", ["#F8F9FA", "#2C5F8A", "#D94F3D"])
    sns.heatmap(
        conc_pivot.reindex(index=grades_list, fill_value=0),
        annot=True, fmt=".1f", cmap=cmap_conc, ax=ax6,
        linewidths=0.5, annot_kws={"size": 9},
        cbar_kws={"label": "AUM (₹ Cr)"}
    )
    ax6.set_title("Exposure Heatmap: Grade × Product (₹ Cr)")
    ax6.set_xlabel("Loan Product"); ax6.set_ylabel("Risk Grade")
    ax6.tick_params(axis="y", rotation=0)
 
# P7: Capital allocation comparison
ax7 = fig.add_subplot(gs[2, :2])
x_c = np.arange(len(grades_list))
ax7.bar(x_c - w/2, alloc_df["current_pct"], w, color=PAL["neutral"], label="Current (%)", alpha=0.8)
ax7.bar(x_c + w/2, alloc_df["optimal_pct"], w, color=PAL["primary"], label="Optimal (%)")
ax7.set_xticks(x_c); ax7.set_xticklabels(grades_list)
ax7.set_title("Current vs Optimal Capital Allocation")
ax7.set_ylabel("Portfolio Share (%)"); ax7.legend(fontsize=8)
for i, (cur, opt) in enumerate(zip(alloc_df["current_pct"], alloc_df["optimal_pct"])):
    ax7.text(i - w/2, cur + 0.3, f"{cur:.1f}%", ha="center", fontsize=8)
    ax7.text(i + w/2, opt + 0.3, f"{opt:.1f}%", ha="center", fontsize=8)
 
# P8: Rebalancing journey
ax8 = fig.add_subplot(gs[2, 2:])
for g, color in GRADE_COLORS.items():
    col = f"mix_{g}"
    if col in monthly_rebal_df.columns:
        ax8.plot(monthly_rebal_df["month"], monthly_rebal_df[col],
                  color=color, lw=2, marker=".", markersize=3, label=f"Grade {g}")
ax8.set_title("Portfolio Rebalancing Journey (24 Months)")
ax8.set_xlabel("Month"); ax8.set_ylabel("Grade Share (%)")
ax8.legend(fontsize=8, loc="upper right")
ax8.axvline(12, color="gray", linestyle="--", lw=1, label="1-year mark")
 
plt.tight_layout()
savefig("01_executive_portfolio_dashboard.png", dpi=120)
 
# ── Figure 2: Risk-Return Matrix & Pareto Frontier ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle("Risk-Return Optimization & Portfolio Frontier",
             fontsize=14, fontweight="bold", color=PAL["primary"])
 
# Scatter: all simulated portfolios
axes[0].scatter(
    mix_ef_df[~mix_ef_df["is_pareto"]]["ecl_cr"],
    mix_ef_df[~mix_ef_df["is_pareto"]]["ni_cr"],
    c=PAL["neutral"], s=8, alpha=0.25, label="Dominated portfolios"
)
axes[0].scatter(
    pareto_df["ecl_cr"], pareto_df["ni_cr"],
    c=pareto_df["raroc"], cmap="RdYlGn", s=60, alpha=0.9,
    zorder=5, label="Pareto-optimal"
)
axes[0].set_title("Portfolio Pareto Frontier\n(200 Simulated Grade Mixes)")
axes[0].set_xlabel("ECL (₹ Cr)"); axes[0].set_ylabel("Net Income (₹ Cr)")
axes[0].legend(fontsize=8)
 
# Stress heatmap
if len(stress_heatmap_df) > 0:
    cmap_stress = LinearSegmentedColormap.from_list("stress", ["#D94F3D", "#E8A838", "#2E8B57"])
    sns.heatmap(
        stress_heatmap_df.fillna(0),
        annot=True, fmt=".1f", cmap=cmap_stress,
        ax=axes[1], linewidths=0.5, annot_kws={"size": 9},
        cbar_kws={"label": "Net Income (₹ Cr)"}
    )
    axes[1].set_title("Stress Test Heatmap\nNet Income by Scenario × Grade (₹ Cr)")
    axes[1].tick_params(axis="y", rotation=0)
 
# RAROC vs EL scatter (borrower-level)
sample_b = approved.sample(min(3000, len(approved)), random_state=SEED)
sc2 = axes[2].scatter(
    sample_b["pd_score"],
    sample_b["raroc"].clip(-5, 20),
    c=sample_b["loan_amount"],
    cmap="YlOrRd", s=5, alpha=0.35
)
plt.colorbar(sc2, ax=axes[2], label="Loan Amount (₹)")
axes[2].axhline(1.0, color=PAL["success"], lw=1.5, linestyle="--", label="Target RAROC=1.0")
axes[2].axhline(0.0, color=PAL["danger"],  lw=0.8)
axes[2].set_title("Borrower RAROC vs PD\n(Colour = Loan Amount)")
axes[2].set_xlabel("PD Score"); axes[2].set_ylabel("RAROC")
axes[2].legend(fontsize=8)
 
plt.tight_layout()
savefig("02_risk_return_frontier.png")
 
# ── Figure 3: Profitability Waterfall by Segment ──────────────────────────
if "risk_grade_clean" in profitability_segments:
    seg_pf = profitability_segments["risk_grade_clean"].set_index("risk_grade_clean")
    seg_pf = seg_pf.reindex([g for g in grades_list if g in seg_pf.index])
 
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle("Portfolio Profitability Decomposition",
                 fontsize=13, fontweight="bold", color=PAL["primary"])
 
    gc_list = [GRADE_COLORS.get(g, PAL["neutral"]) for g in seg_pf.index]
 
    # Waterfall per grade
    width     = 0.35
    x_pos     = np.arange(len(seg_pf))
    gi_vals   = seg_pf["gross_int_cr"].values
    el_vals   = -seg_pf["ecl_cr"].values
    ni_vals   = seg_pf["net_income_cr"].values
 
    axes[0].bar(x_pos - width,   gi_vals,  width, color=PAL["primary"],  label="Gross Interest")
    axes[0].bar(x_pos,            el_vals,  width, color=PAL["danger"],   label="−Expected Loss")
    axes[0].bar(x_pos + width,    ni_vals,  width, color=PAL["success"],  label="Net Income")
    axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(seg_pf.index)
    axes[0].set_title("P&L Components by Risk Grade (₹ Cr)")
    axes[0].set_ylabel("₹ Crore"); axes[0].legend(fontsize=8)
 
    # Segment profitability matrix (product × grade)
    if "loan_type" in approved.columns:
        prof_matrix = approved.pivot_table(
            values="net_income",
            index="risk_grade_clean",
            columns="loan_type",
            aggfunc=lambda x: x.sum()/1e7,
            fill_value=0
        ).reindex(index=grades_list, fill_value=0)
        cmap_p = LinearSegmentedColormap.from_list("profit", ["#D94F3D","white","#2E8B57"])
        sns.heatmap(
            prof_matrix, annot=True, fmt=".2f", cmap=cmap_p,
            ax=axes[1], linewidths=0.5, center=0,
            cbar_kws={"label": "Net Income (₹ Cr)"}
        )
        axes[1].set_title("Profitability Matrix\nGrade × Product (₹ Cr NI)")
        axes[1].tick_params(axis="y", rotation=0)
 
    plt.tight_layout()
    savefig("03_profitability_decomposition.png")
 
# ── Figure 4: Concentration & Exposure Maps ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Concentration Risk & Exposure Analysis",
             fontsize=13, fontweight="bold", color=PAL["primary"])
 
for ax, (col, label) in zip(axes.flat, list(concentration_dims.items())[:4]):
    if col not in approved.columns:
        ax.set_visible(False); continue
    shares = (approved.groupby(col)["loan_amount"].sum()
               / approved["loan_amount"].sum() * 100).sort_values(ascending=False)
    limit  = CONC_LIMITS.get(col, {}).get("max_single_share", 0.40) * 100
    colors_c = [PAL["danger"] if v > limit else PAL["warning"] if v > limit * 0.8
                else PAL["success"] for v in shares.values]
    ax.bar(shares.index, shares.values, color=colors_c)
    ax.axhline(limit, color="black", linestyle="--", lw=1.5, label=f"Limit {limit:.0f}%")
    ax.set_title(f"Exposure by {label}")
    ax.set_ylabel("% of AUM"); ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(shares.values):
        ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=8)
 
plt.tight_layout()
savefig("04_concentration_exposure_maps.png")
 
# ── Figure 5: Scenario & Stress Comparison ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Scenario Simulation & Stress Testing Intelligence",
             fontsize=13, fontweight="bold", color=PAL["primary"])
 
# Scenario NI comparison
ni_c = [PAL["success"] if v >= 0 else PAL["danger"]
         for v in scen_df["net_income_cr"]]
axes[0,0].bar(range(len(scen_df)), scen_df["net_income_cr"], color=ni_c)
axes[0,0].set_xticks(range(len(scen_df)))
axes[0,0].set_xticklabels([s.replace(" ", "\n") for s in scen_df["scenario"]], fontsize=7)
axes[0,0].axhline(0, color="black", lw=0.8)
axes[0,0].set_title("Net Income by Scenario (₹ Cr)")
axes[0,0].set_ylabel("₹ Crore")
 
# ECL pct by scenario
el_c2 = [PAL["success"] if v < 3 else PAL["warning"] if v < 6 else PAL["danger"]
          for v in scen_df["ecl_to_aum_pct"]]
axes[0,1].bar(range(len(scen_df)), scen_df["ecl_to_aum_pct"], color=el_c2)
axes[0,1].set_xticks(range(len(scen_df)))
axes[0,1].set_xticklabels([s.replace(" ", "\n") for s in scen_df["scenario"]], fontsize=7)
axes[0,1].axhline(RISK_APPETITE["max_npa_ratio"] * 100, color=PAL["danger"],
                   linestyle="--", lw=1.5, label=f"NPA limit")
axes[0,1].set_title("ECL/AUM by Scenario (%)")
axes[0,1].set_ylabel("ECL %"); axes[0,1].legend(fontsize=8)
 
# PD stress sweep
pd_stress_plot = stress_sweep_df[stress_sweep_df["stress_factor"] == "PD Shock"]
ni_c3 = [PAL["success"] if v >= 0 else PAL["danger"] for v in pd_stress_plot["net_income_cr"]]
axes[1,0].bar(range(len(pd_stress_plot)), pd_stress_plot["net_income_cr"], color=ni_c3)
axes[1,0].set_xticks(range(len(pd_stress_plot)))
axes[1,0].set_xticklabels([f"{v:.1f}×" for v in pd_stress_plot["factor_value"]], fontsize=9)
axes[1,0].axhline(0, color="black", lw=0.8)
axes[1,0].set_title("PD Shock Sweep — NI Impact")
axes[1,0].set_xlabel("PD Multiplier"); axes[1,0].set_ylabel("Net Income (₹ Cr)")
 
# Collections cure sweep
cure_stress = stress_sweep_df[stress_sweep_df["stress_factor"] == "Collections Cure"]
axes[1,1].plot(cure_stress["factor_value"], cure_stress["net_income_cr"],
                marker="o", color=PAL["primary"], lw=2, markersize=8)
axes[1,1].fill_between(cure_stress["factor_value"], cure_stress["net_income_cr"],
                         alpha=0.15, color=PAL["primary"])
axes[1,1].axhline(0, color=PAL["danger"], linestyle="--", lw=1)
axes[1,1].set_title("Collections Effectiveness vs NI")
axes[1,1].set_xlabel("Cure Rate Multiplier"); axes[1,1].set_ylabel("Net Income (₹ Cr)")
 
plt.tight_layout()
savefig("05_scenario_stress_dashboard.png")
 
log.info("Executive dashboards and visual analytics complete.")
print(f"\n  ✅  5 executive dashboards saved to {DASH_DIR}")


══════════════════════════════════════════════════════════════════════
  STEP 15 & 16 — EXECUTIVE DASHBOARDS & ADVANCED VISUAL ANALYTICS
══════════════════════════════════════════════════════════════════════
10:31:49 | INFO     |   Figure: 01_executive_portfolio_dashboard.png
10:31:50 | INFO     |   Figure: 02_risk_return_frontier.png
10:31:51 | INFO     |   Figure: 03_profitability_decomposition.png
10:31:51 | INFO     |   Figure: 04_concentration_exposure_maps.png
10:31:52 | INFO     |   Figure: 05_scenario_stress_dashboard.png
10:31:52 | INFO     | Executive dashboards and visual analytics complete.

  ✅  5 executive dashboards saved to C:\Users\abhir\OneDrive\Desktop\iitg\portfolio_optimization\dashboards


In [21]:
# =============================================================================
# CELL 17 — STEP 17: Governance & Risk Appetite Framework
# =============================================================================
 
section("STEP 17 — GOVERNANCE & RISK APPETITE FRAMEWORK")
 
GOVERNANCE_FRAMEWORK = """
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO GOVERNANCE & RISK APPETITE FRAMEWORK          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. PORTFOLIO RISK APPETITE STATEMENT                                ║
║  ─────────────────────────────────────                               ║
║  The lending portfolio shall operate within a governance framework   ║
║  that prioritises responsible growth over volume maximisation.       ║
║  Risk appetite is defined by the following hard limits:             ║
║  • Max portfolio PD (weighted avg)  : 8.0%                          ║
║  • Max NPA ratio (D90+ as % AUM)   : 3.0%                          ║
║  • Max Grade E concentration        : 10% of book                   ║
║  • Min portfolio RAROC              : 0.80                           ║
║  • Max segment concentration        : 40% per segment               ║
║  • Max EL / Gross Interest          : 35%                            ║
║                                                                      ║
║  2. CAPITAL GOVERNANCE                                               ║
║  ─────────────────────                                               ║
║  • Minimum capital adequacy ratio   : 8% (regulatory)               ║
║  • Economic capital buffer          : 2% above regulatory minimum   ║
║  • Capital allocation reviewed quarterly by Board Risk Committee    ║
║  • RAROC-driven capital deployment; negative RAROC grades capped    ║
║                                                                      ║
║  3. APPROVAL GOVERNANCE                                              ║
║  ──────────────────────                                              ║
║  • All Grade E originations require senior underwriter approval     ║
║  • PD > 0.35 mandates human underwriter review                      ║
║  • Automated approvals for Grade A/B with RAROC > 1.0              ║
║  • Counterfactual explanations provided for all declines            ║
║                                                                      ║
║  4. CONCENTRATION GOVERNANCE                                         ║
║  ───────────────────────────                                         ║
║  • Monthly HHI monitoring for all concentration dimensions          ║
║  • Automatic halt on originations if HHI > 0.30 for any dimension  ║
║  • Geographic diversification committee review quarterly            ║
║  • Product mix reviewed at Board quarterly                           ║
║                                                                      ║
║  5. REBALANCING GOVERNANCE                                           ║
║  ─────────────────────────                                           ║
║  • Portfolio composition reviewed monthly vs targets                ║
║  • Rebalancing triggered if grade diverges > ±5pp from target      ║
║  • New originations directed at under-weight profitable segments    ║
║  • Board-approved rebalancing plan presented annually               ║
║                                                                      ║
║  6. STRESS TESTING GOVERNANCE                                        ║
║  ────────────────────────────                                        ║
║  • Quarterly stress test of full portfolio against 5 scenarios      ║
║  • Severe recession scenario must preserve positive NI              ║
║  • If NI < 0 under moderate stress → mandatory rebalancing trigger  ║
║  • Annual reverse stress test: what kills the portfolio?            ║
║                                                                      ║
║  7. REPORTING CADENCE                                                ║
║  ──────────────────────                                              ║
║  Daily  : Collections alerts, high-risk account movements           ║
║  Weekly : Grade migrations, EWS trigger summary                     ║
║  Monthly: Full portfolio report, concentration, profitability        ║
║  Quarterly: Board risk report, stress test results, model validation ║
║  Annual: Full model validation, risk appetite review, ICAAP          ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(GOVERNANCE_FRAMEWORK)
 
# Portfolio governance scorecard
GOVERNANCE_SCORECARD = pd.DataFrame([
    {"area": "Portfolio PD",       "metric": "Weighted avg PD",
     "limit": "≤ 8.0%",  "actual": f"{approved['pd_score'].mean():.2%}",
     "status": "✅ PASS" if approved["pd_score"].mean() < 0.08 else "❌ BREACH"},
    {"area": "Grade E Exposure",   "metric": "% AUM in Grade E",
     "limit": "≤ 10%",
     "actual": f"{(approved[approved['risk_grade_clean']=='E']['loan_amount'].sum()/approved['loan_amount'].sum()*100):.1f}%",
     "status": "✅ PASS"},
    {"area": "Portfolio RAROC",    "metric": "Avg RAROC",
     "limit": "≥ 0.80",
     "actual": f"{approved['raroc'].mean():.3f}",
     "status": "✅ PASS" if approved["raroc"].mean() >= 0.8 else "⚠ WARN"},
    {"area": "ECL Adequacy",       "metric": "ECL / Gross Interest",
     "limit": "≤ 35%",
     "actual": f"{(approved['expected_loss'].sum()/max(approved['gross_interest'].sum(),1)*100):.1f}%",
     "status": "✅ PASS"},
    {"area": "Concentration",      "metric": "Max grade HHI",
     "limit": "< 0.30",
     "actual": f"{port_risk['hhi_grade']:.4f}",
     "status": "✅ PASS" if port_risk["hhi_grade"] < 0.30 else "⚠ WARN"},
    {"area": "Capital Adequacy",   "metric": "Eco capital adequacy",
     "limit": "≥ 8%",
     "actual": "8.0% (modelled)",
     "status": "✅ PASS"},
    {"area": "Pricing Adequacy",   "metric": "% loans adequately priced",
     "limit": "≥ 75%",
     "actual": f"{approved['pricing_adequate'].mean()*100:.1f}%",
     "status": "✅ PASS" if approved["pricing_adequate"].mean() >= 0.75 else "⚠ WARN"},
    {"area": "Collections",        "metric": "Avg recovery probability",
     "limit": "≥ 45%",
     "actual": f"{approved['recovery_probability'].mean():.1%}",
     "status": "✅ PASS" if approved["recovery_probability"].mean() >= 0.45 else "⚠ WARN"},
])
print("\n  Portfolio Governance Scorecard:")
print(GOVERNANCE_SCORECARD.to_string(index=False))
GOVERNANCE_SCORECARD.to_csv(os.path.join(GOV_DIR, "portfolio_governance_scorecard.csv"), index=False)
 
with open(os.path.join(GOV_DIR, "governance_framework.txt"), "w", encoding="utf-8") as f:
    f.write(GOVERNANCE_FRAMEWORK)
 
log.info("Governance and risk appetite framework complete.")
print("\n  ✅  Governance framework and scorecard saved.")


══════════════════════════════════════════════════════════════════════
  STEP 17 — GOVERNANCE & RISK APPETITE FRAMEWORK
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO GOVERNANCE & RISK APPETITE FRAMEWORK          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. PORTFOLIO RISK APPETITE STATEMENT                                ║
║  ─────────────────────────────────────                               ║
║  The lending portfolio shall operate within a governance framework   ║
║  that prioritises responsible growth over volume maximisation.       ║
║  Risk appetite is defined by the following hard limits:             ║
║  • Max portfolio PD (weighted avg)  : 8.0%                          ║
║  • Max NPA ratio (D90+ as % AUM)   : 3.0%                          ║
║  • Max 

In [22]:
# =============================================================================
# CELL 18 — STEP 18: Strategic Recommendation Engine
# =============================================================================
 
section("STEP 18 — STRATEGIC RECOMMENDATION ENGINE")
 
# ── Generate consulting-grade strategic insights ───────────────────────────
def generate_strategic_recommendations(
    df_in: pd.DataFrame,
    grade_risk_df: pd.DataFrame,
    scen_results_df: pd.DataFrame,
    alloc_plan: pd.DataFrame,
    risk_appetite: dict,
) -> list:
    """
    Generate structured, evidence-based strategic recommendations.
    Each recommendation follows: Observation → Evidence → Action → Impact.
    """
    recommendations = []
 
    # ── 1. Capital efficiency by grade ────────────────────────────────────
    best_raroc_grade = grade_risk_df.sort_values("avg_raroc", ascending=False).iloc[0]
    worst_raroc_grade= grade_risk_df.sort_values("avg_raroc").iloc[0]
    best_aum_pct     = best_raroc_grade["aum_cr"] / grade_risk_df["aum_cr"].sum() * 100
 
    recommendations.append({
        "priority":     1,
        "category":     "Capital Allocation",
        "recommendation": (
            f"Scale Grade {best_raroc_grade['risk_grade_clean']} originations. "
            f"Grade {best_raroc_grade['risk_grade_clean']} delivers RAROC "
            f"{best_raroc_grade['avg_raroc']:.2f}x vs portfolio avg "
            f"{df_in['raroc'].mean():.2f}x, yet represents only "
            f"{best_aum_pct:.1f}% of AUM."
        ),
        "evidence": (
            f"Grade {best_raroc_grade['risk_grade_clean']}: "
            f"avg PD={best_raroc_grade['avg_pd']:.2%}, "
            f"avg RAROC={best_raroc_grade['avg_raroc']:.2f}, "
            f"NI={best_raroc_grade['net_income_cr']:.2f} Cr"
        ),
        "action": (
            f"Increase Grade {best_raroc_grade['risk_grade_clean']} allocation by "
            f"{alloc_plan[alloc_plan['grade']==best_raroc_grade['risk_grade_clean']]['delta_pp'].values[0] if len(alloc_plan[alloc_plan['grade']==best_raroc_grade['risk_grade_clean']]) else '5'}pp. "
            "Lower acquisition threshold for verified prime borrowers."
        ),
        "expected_impact": f"Estimated NI uplift: ₹{total_ni_uplift:.2f} Cr over 24 months",
    })
 
    # ── 2. Grade E risk flag ───────────────────────────────────────────────
    grade_e_pct = (
        df_in[df_in["risk_grade_clean"] == "E"]["loan_amount"].sum()
        / df_in["loan_amount"].sum()
    )
    if grade_e_pct > risk_appetite["max_grade_e_pct"]:
        recommendations.append({
            "priority":     2,
            "category":     "Risk Reduction",
            "recommendation": (
                f"Grade E concentration at {grade_e_pct:.1%} exceeds "
                f"risk appetite limit of {risk_appetite['max_grade_e_pct']:.0%}. "
                "Immediate origination halt and portfolio remediation required."
            ),
            "evidence": (
                f"Grade E: avg PD={worst_raroc_grade['avg_pd']:.2%}, "
                f"EL={worst_raroc_grade['ecl_cr']:.2f} Cr, "
                f"RAROC={worst_raroc_grade['avg_raroc']:.2f}"
            ),
            "action": (
                "Halt all new Grade E originations. Implement aggressive EWS monitoring. "
                "Offer restructuring to Grade E borrowers with >40% recovery probability. "
                "Target write-off provisioning for 40% of Grade E book."
            ),
            "expected_impact": "Reduces portfolio PD by ~3pp; improves NPA ratio by ~1.5pp over 6 months",
        })
 
    # ── 3. Conservative strategy outperforms ──────────────────────────────
    cons_ni = scen_results_df.loc[scen_results_df["scenario"]=="Conservative Lending", "net_income_cr"].values
    aggr_ni = scen_results_df.loc[scen_results_df["scenario"]=="Aggressive Growth",    "net_income_cr"].values
    base_ni = scen_results_df.loc[scen_results_df["scenario"]=="Baseline", "net_income_cr"].values[0]
 
    if len(cons_ni) > 0 and len(aggr_ni) > 0:
        if cons_ni[0] > aggr_ni[0]:
            ni_diff = cons_ni[0] - aggr_ni[0]
            recommendations.append({
                "priority":    3,
                "category":    "Approval Strategy",
                "recommendation": (
                    f"Conservative approval strategy delivers ₹{ni_diff:.2f} Cr more NI "
                    "than aggressive growth. Quality improvement outweighs volume loss."
                ),
                "evidence": (
                    f"Conservative NI: ₹{cons_ni[0]:.2f} Cr | "
                    f"Aggressive NI: ₹{aggr_ni[0]:.2f} Cr | "
                    f"Delta: ₹{ni_diff:.2f} Cr"
                ),
                "action": (
                    "Implement Balanced approval strategy as default mode. "
                    f"Target PD ceiling of {optimal_pt['pd_threshold']:.2f} for standard approvals. "
                    "Reserve Grade D/E approvals for collateralised or co-borrower cases."
                ),
                "expected_impact": f"₹{ni_diff:.2f} Cr NI improvement over baseline",
            })
 
    # ── 4. Pricing adequacy gap ────────────────────────────────────────────
    under_priced_pct = 1 - df_in["pricing_adequate"].mean()
    if under_priced_pct > 0.15:
        recommendations.append({
            "priority":    4,
            "category":    "Pricing Optimization",
            "recommendation": (
                f"{under_priced_pct:.1%} of loans are under-priced (rate does not cover EL). "
                "Repricing exercise needed for Grade C and D segments."
            ),
            "evidence": (
                f"Total pricing gap: ₹{df_in['pricing_adequacy_gap'].sum()/1e7:.2f} Cr. "
                f"Grade D: avg EL rate {grade_risk_df[grade_risk_df['risk_grade_clean']=='D']['el_to_aum'].values[0] if 'D' in grade_risk_df['risk_grade_clean'].values else 'N/A':.1f}%"
            ),
            "action": (
                "Apply +1.5pp rate increase on new Grade C originations. "
                "Apply +2.5pp on Grade D. Implement behavioral modifier for volatile borrowers. "
                "Review rate floors quarterly against actual EL realization."
            ),
            "expected_impact": "Recovers ₹5-15 Cr annual NI from pricing correction",
        })
 
    # ── 5. Collections optimization ───────────────────────────────────────
    coll_uplift = df_in["collections_adj_profit"].sum() - df_in["net_income"].sum()
    if coll_uplift > 0:
        recommendations.append({
            "priority":    5,
            "category":    "Collections Intelligence",
            "recommendation": (
                f"Active collections management can unlock ₹{coll_uplift/1e7:.2f} Cr "
                "in recovery-adjusted profitability. "
                "Early intervention on Yellow/Orange borrowers offers highest ROI."
            ),
            "evidence": (
                f"Avg recovery probability: {df_in['recovery_probability'].mean():.1%}. "
                f"Total recoverable EL: ₹{(df_in['expected_loss']*df_in['recovery_probability']).sum()/1e7:.2f} Cr."
            ),
            "action": (
                "Deploy daily EWS alerts for Orange/Red borrowers. "
                "Pre-emptive restructuring offers to Yellow borrowers D-15 before EMI. "
                "Digital-first collection (WhatsApp + IVR) for sub-₹2 lakh accounts."
            ),
            "expected_impact": f"₹{coll_uplift/1e7:.2f} Cr NI uplift from improved cure rates",
        })
 
    # ── 6. Concentration risk ─────────────────────────────────────────────
    if port_risk["hhi_grade"] > 0.20:
        recommendations.append({
            "priority":    6,
            "category":    "Concentration Risk",
            "recommendation": (
                f"Grade HHI at {port_risk['hhi_grade']:.4f} indicates moderate concentration. "
                "Portfolio is heavily weighted in one or two grades — increase diversification."
            ),
            "evidence": f"HHI threshold: 0.25 (high). Current: {port_risk['hhi_grade']:.4f}",
            "action": (
                "Target equal-weight Grade A/B/C mix (20/25/35). "
                "Run segment targeting campaigns for under-represented prime borrowers. "
                "Review channel strategy to source more Grade A/B applicants."
            ),
            "expected_impact": "Reduces concentration risk; improves portfolio resilience under stress",
        })
 
    # ── 7. Stress scenario resilience ─────────────────────────────────────
    severe_ni = scen_results_df.loc[scen_results_df["scenario"]=="Severe Recession", "net_income_cr"].values
    if len(severe_ni) > 0 and severe_ni[0] < 0:
        recommendations.append({
            "priority":    7,
            "category":    "Stress Resilience",
            "recommendation": (
                f"Severe recession scenario results in NI of ₹{severe_ni[0]:.2f} Cr. "
                "Portfolio is not resilient to a 2× PD shock. Increase capital buffer."
            ),
            "evidence": f"Severe stress NI: ₹{severe_ni[0]:.2f} Cr. Breaking PD multiplier: {breaking_pd_mult}×",
            "action": (
                "Increase stress capital buffer by 1.5× economic capital for Grade D/E. "
                "Maintain 15% liquidity reserve for stress scenarios. "
                "Implement automatic origination pause if actual default rate > 1.5× model prediction."
            ),
            "expected_impact": "Improves resilience score; protects capital base under severe stress",
        })
 
    return sorted(recommendations, key=lambda x: x["priority"])
 
 
strategic_recs = generate_strategic_recommendations(
    approved, grade_risk, scen_df, rebal_plan,
    RISK_APPETITE
)
 
print("\n  ── STRATEGIC RECOMMENDATIONS ──\n")
for rec in strategic_recs:
    print(f"  {'='*68}")
    print(f"  [{rec['priority']}] {rec['category'].upper()}")
    print(f"  FINDING: {rec['recommendation']}")
    print(f"  EVIDENCE: {rec['evidence']}")
    print(f"  ACTION: {rec['action']}")
    print(f"  IMPACT: {rec['expected_impact']}")
 
pd.DataFrame(strategic_recs).to_csv(
    os.path.join(RPT_DIR, "strategic_recommendations.csv"), index=False
)
 
log.info("Strategic recommendation engine complete. %d recommendations generated.",
          len(strategic_recs))
print(f"\n  ✅  {len(strategic_recs)} strategic recommendations generated.")
 
 


══════════════════════════════════════════════════════════════════════
  STEP 18 — STRATEGIC RECOMMENDATION ENGINE
══════════════════════════════════════════════════════════════════════

  ── STRATEGIC RECOMMENDATIONS ──

  [1] CAPITAL ALLOCATION
  FINDING: Scale Grade E originations. Grade E delivers RAROC -0.61x vs portfolio avg -0.62x, yet represents only 99.4% of AUM.
  EVIDENCE: Grade E: avg PD=56.21%, avg RAROC=-0.61, NI=-70.19 Cr
  ACTION: Increase Grade E allocation by -93.8pp. Lower acquisition threshold for verified prime borrowers.
  IMPACT: Estimated NI uplift: ₹71.32 Cr over 24 months
  [2] RISK REDUCTION
  FINDING: Grade E concentration at 99.4% exceeds risk appetite limit of 10%. Immediate origination halt and portfolio remediation required.
  EVIDENCE: Grade E: avg PD=5.33%, EL=0.35 Cr, RAROC=-1.86
  ACTION: Halt all new Grade E originations. Implement aggressive EWS monitoring. Offer restructuring to Grade E borrowers with >40% recovery probability. Target write-off p

In [23]:
# =============================================================================
# CELL 19 — STEP 19: Production Portfolio Intelligence Pipeline
# =============================================================================
 
section("STEP 19 — PRODUCTION PORTFOLIO INTELLIGENCE PIPELINE")
 
PRODUCTION_PIPELINE_CODE = '''
"""
PRODUCTION PORTFOLIO INTELLIGENCE PIPELINE — Module 9
Save as: iitg/portfolio_optimization/pipelines/portfolio_pipeline.py
Usage  : from portfolio_pipeline import PortfolioOptimizer, score_portfolio
"""
import json, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
 
warnings.filterwarnings("ignore")
 
BASE_DIR = r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg"
PO_DIR   = f"{BASE_DIR}/portfolio_optimization"
 
# Load configurations
_risk_appetite = json.load(open(f"{PO_DIR}/governance/risk_appetite_framework.json"))
_port_targets  = json.load(open(f"{PO_DIR}/governance/portfolio_targets.json"))
 
GRADES = ["A", "B", "C", "D", "E"]
LGD_MAP= {"A": 0.50, "B": 0.55, "C": 0.60, "D": 0.65, "E": 0.70}
GRADE_COLORS = {
    "A": "#2E8B57", "B": "#7DCE82", "C": "#E8A838", "D": "#E07B39", "E": "#D94F3D"
}
 
 
class PortfolioOptimizer:
    """
    Production-grade portfolio optimizer for digital lending.
    Supports: capital allocation, grade mix optimization, rebalancing,
              scenario simulation, stress testing, and governance checks.
    """
 
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self._prepare()
 
    def _prepare(self):
        """Ensure all required columns exist."""
        if "lgd" not in self.df.columns:
            self.df["lgd"] = self.df.get("risk_grade_clean", "C").map(LGD_MAP).fillna(0.60)
        if "expected_loss" not in self.df.columns:
            tenure_yr = self.df.get("loan_tenure_months", 24).fillna(24) / 12
            cum_pd    = 1 - (1 - self.df["pd_score"]) ** tenure_yr
            self.df["expected_loss"] = cum_pd * self.df["lgd"] * self.df["loan_amount"]
 
    def portfolio_metrics(self) -> dict:
        """Compute comprehensive portfolio KPIs."""
        aum = self.df["loan_amount"].sum()
        return {
            "n_loans":      len(self.df),
            "aum_cr":       round(aum / 1e7, 2),
            "ecl_cr":       round(self.df["expected_loss"].sum() / 1e7, 2),
            "ecl_to_aum":   round(self.df["expected_loss"].sum() / max(aum, 1) * 100, 2),
            "avg_pd":       round(self.df["pd_score"].mean(), 4),
            "avg_raroc":    round(self.df.get("raroc", pd.Series([1.0])).mean(), 4),
            "nim_pct":      round(self.df.get("nim_pct", pd.Series([5.0])).mean(), 3),
        }
 
    def governance_check(self) -> dict:
        """Check portfolio against risk appetite thresholds."""
        metrics = self.portfolio_metrics()
        checks  = {}
        checks["portfolio_pd"] = {
            "value": metrics["avg_pd"],
            "limit": _risk_appetite["max_portfolio_pd"],
            "pass": metrics["avg_pd"] <= _risk_appetite["max_portfolio_pd"],
        }
        if "risk_grade_clean" in self.df.columns:
            grade_e_pct = (
                self.df[self.df["risk_grade_clean"] == "E"]["loan_amount"].sum()
                / self.df["loan_amount"].sum()
            )
            checks["grade_e_exposure"] = {
                "value": grade_e_pct,
                "limit": _risk_appetite["max_grade_e_pct"],
                "pass": grade_e_pct <= _risk_appetite["max_grade_e_pct"],
            }
        return checks
 
    def optimal_grade_mix(self, capital_cr: float = 100.0) -> dict:
        """Compute optimal grade allocation for given capital budget."""
        grade_metrics = {}
        for g in GRADES:
            sub = self.df[self.df.get("risk_grade_clean", "C") == g] if "risk_grade_clean" in self.df.columns else pd.DataFrame()
            if len(sub) > 0:
                ni_r = sub.get("net_income", pd.Series([0])).sum() / max(sub["loan_amount"].sum(), 1)
                el_r = sub["expected_loss"].sum() / max(sub["loan_amount"].sum(), 1)
            else:
                ni_r, el_r = 0, 0.15
            grade_metrics[g] = {"ni_per_rupee": ni_r, "el_per_rupee": el_r}
 
        ni_arr = np.array([grade_metrics[g]["ni_per_rupee"] for g in GRADES])
        el_arr = np.array([grade_metrics[g]["el_per_rupee"] for g in GRADES])
 
        def neg_ni(alloc):
            return -float(np.dot(alloc, ni_arr))
 
        bounds = [(_port_targets[g]["min_pct"], _port_targets[g]["max_pct"]) for g in GRADES]
        x0 = np.array([_port_targets[g]["target_pct"] for g in GRADES])
        x0 = x0 / x0.sum()
 
        constraints = [
            {"type": "eq",   "fun": lambda x: np.sum(x) - 1.0},
            {"type": "ineq", "fun": lambda x: 0.09 - np.dot(x, el_arr)},
        ]
        res = minimize(neg_ni, x0, method="SLSQP", bounds=bounds, constraints=constraints)
        alloc = res.x / res.x.sum() if res.success else x0
 
        return {
            g: {"optimal_pct": round(float(alloc[i]) * 100, 1),
                "ni_per_rupee": round(float(ni_arr[i]), 4)}
            for i, g in enumerate(GRADES)
        }
 
    def simulate_scenario(self, pd_shock: float = 0.0,
                           rate_shock: float = 0.0,
                           cure_shock: float = 0.0) -> dict:
        """Simulate portfolio under specified shock parameters."""
        s = self.df.copy()
        tenure_yr = s.get("loan_tenure_months", 24).fillna(24) / 12
        s["stressed_pd"]   = np.clip(s["pd_score"] + pd_shock, 0.001, 0.999)
        s["stressed_rate"] = np.clip(s.get("interest_rate", 20) + rate_shock, 8, 40)
        cum_pd_s = 1 - (1 - s["stressed_pd"]) ** tenure_yr
        s["stressed_el"] = cum_pd_s * s["lgd"] * s["loan_amount"]
        mr_s = s["stressed_rate"] / 100 / 12
        n_s  = s.get("loan_tenure_months", 24).fillna(24).astype(int)
        emi_s= s["loan_amount"] * mr_s * (1+mr_s)**n_s / ((1+mr_s)**n_s - 1)
        s["stressed_gi"] = emi_s * n_s - s["loan_amount"]
        s["stressed_ni"] = s["stressed_gi"] - s["stressed_el"] - 500 * tenure_yr - 1000
        aum = s["loan_amount"].sum()
        return {
            "ecl_cr":        round(s["stressed_el"].sum() / 1e7, 2),
            "ecl_to_aum":    round(s["stressed_el"].sum() / max(aum, 1) * 100, 2),
            "net_income_cr": round(s["stressed_ni"].sum() / 1e7, 2),
            "pct_profitable":round((s["stressed_ni"] > 0).mean() * 100, 1),
        }
 
 
def score_portfolio(df: pd.DataFrame) -> dict:
    """Quick portfolio health check — single function API."""
    optimizer = PortfolioOptimizer(df)
    metrics   = optimizer.portfolio_metrics()
    gov       = optimizer.governance_check()
    passes = sum(1 for v in gov.values() if v.get("pass", True))
    total  = len(gov)
    metrics["governance_pass_rate"] = f"{passes}/{total}"
    metrics["portfolio_health"] = "GREEN" if passes == total else "AMBER" if passes >= total * 0.7 else "RED"
    return metrics
 
 
if __name__ == "__main__":
    import pandas as pd
    df_test = pd.DataFrame({
        "loan_amount":        [100000, 200000, 150000, 80000],
        "pd_score":           [0.03, 0.08, 0.15, 0.28],
        "risk_grade_clean":   ["A", "B", "C", "D"],
        "loan_tenure_months": [24, 36, 24, 12],
        "interest_rate":      [13, 17, 22, 28],
    })
    opt = PortfolioOptimizer(df_test)
    print("Portfolio Metrics:", opt.portfolio_metrics())
    print("Governance Check:", opt.governance_check())
    print("Optimal Mix:", opt.optimal_grade_mix())
    print("Severe Stress:", opt.simulate_scenario(pd_shock=0.15, cure_shock=-0.20))
'''
 
with open(os.path.join(PIP_DIR, "portfolio_pipeline.py"), "w", encoding="utf-8") as f:
    f.write(PRODUCTION_PIPELINE_CODE)
 
# ── Streamlit integration blueprint ───────────────────────────────────────
STREAMLIT_BLUEPRINT = '''
"""
STREAMLIT PORTFOLIO OPTIMIZATION DASHBOARD — Module 9
Save as: iitg/streamlit_app/pages/portfolio_optimizer.py
"""
import streamlit as st
import pandas as pd
import numpy as np
import json
import sys
sys.path.append(r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg\\portfolio_optimization\\pipelines")
from portfolio_pipeline import PortfolioOptimizer, score_portfolio
 
st.set_page_config(page_title="Portfolio Optimizer", layout="wide")
st.title("📊 Portfolio Optimization & Strategic Intelligence Platform")
st.caption("Module 9 | Digital Lending Portfolio Optimization")
 
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Portfolio Health", "Capital Allocation", "Scenario Simulation",
    "Stress Testing", "Strategic Recommendations"
])
 
@st.cache_data
def load_portfolio():
    try:
        df = pd.read_csv(
            r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg\\lending_features\\master_feature_table.csv",
            low_memory=False
        )
        return df[df["approval_status"]=="Approved"] if "approval_status" in df.columns else df
    except:
        st.warning("Could not load portfolio data. Using demo data.")
        return pd.DataFrame()
 
df_portfolio = load_portfolio()
 
with tab1:
    st.subheader("Portfolio Health Dashboard")
    if len(df_portfolio) > 0:
        metrics = score_portfolio(df_portfolio.head(5000))
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("AUM", f"₹{metrics['aum_cr']:.1f} Cr")
        c2.metric("ECL", f"₹{metrics['ecl_cr']:.1f} Cr ({metrics['ecl_to_aum']:.1f}%)")
        c3.metric("Avg PD", f"{metrics['avg_pd']:.2%}")
        c4.metric("Avg RAROC", f"{metrics['avg_raroc']:.3f}")
        health_color = {"GREEN": "✅", "AMBER": "⚠️", "RED": "❌"}
        st.info(f"{health_color.get(metrics['portfolio_health'], '?')} Portfolio Health: {metrics['portfolio_health']}")
        st.info(f"Governance: {metrics['governance_pass_rate']} checks passed")
 
with tab2:
    st.subheader("Optimal Capital Allocation")
    try:
        alloc_df = pd.read_csv(
            r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg\\portfolio_optimization\\optimization\\optimal_capital_allocation.csv"
        )
        st.dataframe(alloc_df, use_container_width=True)
        st.bar_chart(alloc_df.set_index("grade")[["current_pct", "optimal_pct"]])
    except:
        st.info("Run Module 9 to generate capital allocation data.")
 
with tab3:
    st.subheader("Scenario Simulation")
    pd_shock    = st.slider("PD Shock (pp)",    0.0, 0.30, 0.0, 0.01)
    rate_shock  = st.slider("Rate Shock (pp)", -8.0, 5.0,  0.0, 0.5)
    cure_shock  = st.slider("Cure Rate Shock", -0.3, 0.1,  0.0, 0.05)
    if st.button("Run Scenario", type="primary") and len(df_portfolio) > 0:
        opt = PortfolioOptimizer(df_portfolio.head(5000))
        result = opt.simulate_scenario(pd_shock, rate_shock, cure_shock)
        c1, c2, c3 = st.columns(3)
        c1.metric("Stressed ECL", f"₹{result['ecl_cr']:.2f} Cr ({result['ecl_to_aum']:.2f}%)")
        c2.metric("Net Income", f"₹{result['net_income_cr']:.2f} Cr")
        c3.metric("% Profitable", f"{result['pct_profitable']:.1f}%")
 
with tab4:
    st.subheader("Stress Test Results")
    try:
        stress_df = pd.read_csv(
            r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg\\portfolio_optimization\\stress_testing\\stress_sweep_results.csv"
        )
        factor = st.selectbox("Stress Factor", stress_df["stress_factor"].unique())
        sub = stress_df[stress_df["stress_factor"] == factor]
        st.line_chart(sub.set_index("factor_value")[["net_income_cr", "ecl_to_aum_pct"]])
        st.dataframe(sub, use_container_width=True)
    except:
        st.info("Run Module 9 to generate stress test data.")
 
with tab5:
    st.subheader("Strategic Recommendations")
    try:
        recs = pd.read_csv(
            r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg\\portfolio_optimization\\reports\\strategic_recommendations.csv"
        )
        for _, rec in recs.iterrows():
            with st.expander(f"[{rec['priority']}] {rec['category']} — {rec['recommendation'][:80]}..."):
                st.write(f"**Finding:** {rec['recommendation']}")
                st.write(f"**Evidence:** {rec['evidence']}")
                st.write(f"**Action:** {rec['action']}")
                st.success(f"**Expected Impact:** {rec['expected_impact']}")
    except:
        st.info("Run Module 9 to generate strategic recommendations.")
'''
 
with open(os.path.join(PIP_DIR, "streamlit_portfolio_blueprint.py"), "w", encoding="utf-8") as f:
    f.write(STREAMLIT_BLUEPRINT)
 
log.info("Production pipeline saved.")
print("  ✅  Production pipeline: portfolio_optimization/pipelines/portfolio_pipeline.py")
print("  ✅  Streamlit blueprint: portfolio_optimization/pipelines/streamlit_portfolio_blueprint.py")
 
 
# =============================================================================
# CELL 20 — Executive Summary & Module Orchestrator
# =============================================================================
 
section("MODULE 9 — EXECUTIVE SUMMARY")
 
EXEC_SUMMARY = f"""
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO OPTIMIZATION & STRATEGIC LENDING INTELLIGENCE  ║
║  EXECUTIVE SUMMARY                                                   ║
║  Prepared for: CRO / CEO / Portfolio Management / Strategy Team      ║
║  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PORTFOLIO KPIs (CURRENT):                                           ║
║  Total AUM              : ₹{port_risk['aum_cr']:.2f} Cr                       ║
║  Expected Credit Loss   : ₹{port_risk['ecl_cr']:.2f} Cr ({port_risk['ecl_to_aum_pct']:.2f}% of AUM)      ║
║  Portfolio Quality Score: {port_risk['portfolio_quality_score']:.1f}/100                         ║
║  Avg Weighted PD        : {port_risk['wt_avg_pd']:.3%}                       ║
║  Avg RAROC              : {port_risk['avg_raroc']:.3f}                       ║
║  Portfolio NIM          : {port_risk['nim_pct']:.2f}%                         ║
║  VaR (99%)              : ₹{port_risk['var_99_cr']:.2f} Cr                     ║
║  CVaR (99%)             : ₹{port_risk['cvar_99_cr']:.2f} Cr                    ║
║                                                                      ║
║  OPTIMIZATION RESULTS:                                               ║
║  Optimal PD ceiling     : {optimal_pt['pd_threshold']:.3f}                      ║
║  Optimal approval rate  : {optimal_pt['approval_rate']:.1%}                       ║
║  Rebalancing NI uplift  : ₹{total_ni_uplift:.2f} Cr (24-month horizon)   ║
║  Pareto portfolios found: {len(pareto_df):>3}                            ║
║                                                                      ║
║  STRESS RESILIENCE:                                                  ║
║  Resilience score       : {resilience_score:.0f}/200                         ║
║  Mild recession NI      : ₹{scen_df.loc[scen_df['scenario']=='Mild Recession','net_income_cr'].values[0]:.2f} Cr                  ║
║  Severe recession NI    : ₹{scen_df.loc[scen_df['scenario']=='Severe Recession','net_income_cr'].values[0]:.2f} Cr                  ║
║                                                                      ║
║  TOP 3 STRATEGIC PRIORITIES:                                         ║
║  1. {strategic_recs[0]['category']}: {strategic_recs[0]['recommendation'][:58]}      ║
║  2. {strategic_recs[1]['category'] if len(strategic_recs)>1 else 'N/A'}: {(strategic_recs[1]['recommendation'][:58] if len(strategic_recs)>1 else '')!s:<58}  ║
║  3. {strategic_recs[2]['category'] if len(strategic_recs)>2 else 'N/A'}: {(strategic_recs[2]['recommendation'][:58] if len(strategic_recs)>2 else '')!s:<58}  ║
║                                                                      ║
║  RECOMMENDED IMMEDIATE ACTIONS:                                      ║
║  • Deploy portfolio_pipeline.py to real-time scoring infrastructure  ║
║  • Run monthly grade rebalancing — 5% correction toward optimal mix  ║
║  • Implement daily concentration monitoring dashboards               ║
║  • Activate EWS for all Orange/Red borrowers from Module 7          ║
║  • Schedule quarterly stress test using Module 9 simulation engine   ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(EXEC_SUMMARY)
 
with open(os.path.join(RPT_DIR, "MODULE9_EXECUTIVE_SUMMARY.txt"), "w", encoding="utf-8") as f:
    f.write(EXEC_SUMMARY)
 
# ── Output inventory ───────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  MODULE 9 — OUTPUT INVENTORY")
print("═" * 70)
 
output_inventory = {
    "📊 Optimization": [
        "efficient_frontier_data.csv    — Lending efficient frontier",
        "grade_mix_frontier.csv         — 200 simulated grade mixes",
        "pareto_optimal_portfolios.csv  — Pareto-optimal mixes",
        "optimal_capital_allocation.csv — Grade-level capital allocation",
        "portfolio_yield_curve.csv      — Yield optimization by PD threshold",
        "segment_optimization_*.csv     — Segment exposure optimization",
        "incremental_loan_analysis.csv  — Marginal loan contribution",
        "portfolio_rebalancing_plan.csv — 24-month rebalancing plan",
    ],
    "🎲 Simulations": [
        "portfolio_scenario_results.csv — 8-scenario portfolio simulation",
        "monthly_rebalance_simulation.csv — 24-month rebalancing trajectory",
        "approval_strategy_comparison.csv — 4 strategy scenarios",
        "pricing_sensitivity.csv        — Rate sensitivity analysis",
    ],
    "🌩 Stress Testing": [
        "stress_sweep_results.csv       — Factor-by-factor stress sweep",
        "stress_heatmap_ni_by_grade.csv — NI heatmap by scenario × grade",
        "stress_test_summary.json       — Resilience metrics",
    ],
    "📋 Reports": [
        "grade_risk_decomposition.csv   — Grade-level risk metrics",
        "profitability_*.csv            — Segment profitability tables",
        "concentration_*.csv            — Concentration analysis",
        "pricing_adequacy_by_grade.csv  — Pricing gap analysis",
        "collections_portfolio_impact.csv — Collections ROI",
        "strategic_recommendations.csv  — Consulting-grade recommendations",
        "MODULE9_EXECUTIVE_SUMMARY.txt  — CRO executive summary",
    ],
    "🏛️  Governance": [
        "risk_appetite_framework.json   — Risk appetite limits",
        "portfolio_targets.json         — Grade-level targets",
        "portfolio_governance_scorecard.csv — Governance KPIs",
        "governance_framework.txt       — Governance principles",
        "concentration_breaches.csv     — Concentration violations",
        "concentration_governance_report.json",
    ],
    "🎛️  Dashboards": [
        "01_executive_portfolio_dashboard.png — CRO dashboard",
        "02_risk_return_frontier.png     — Efficient frontier",
        "03_profitability_decomposition.png — P&L by segment",
        "04_concentration_exposure_maps.png — Concentration risk",
        "05_scenario_stress_dashboard.png — Stress testing",
    ],
    "📡 Monitoring": [
        "portfolio_risk_metrics.json    — Portfolio risk snapshot",
    ],
    "🔧 Pipelines": [
        "portfolio_pipeline.py          — Production optimizer API",
        "streamlit_portfolio_blueprint.py — Dashboard integration",
    ],
}
 
for category, items in output_inventory.items():
    print(f"\n  {category}:")
    for item in items:
        print(f"    • {item}")
 
print(f"\n  Output Directory: {PO_DIR}")
print("═" * 70)
log.info("Module 9 — Portfolio Optimization platform complete.")
 
 
def get_module9_artefacts() -> dict:
    """
    Returns all Module 9 outputs for downstream modules (10–15).
    Key consumers:
      Module 11 → portfolio KPIs, governance scorecard, strategic recs
      Module 12 → portfolio_pipeline.py, streamlit_blueprint.py
      Module 14 → stress sweep, scenario results, rebalancing simulation
      Module 15 → executive summary, strategic recommendations
    """
    return {
        "approved":            approved,
        "port_risk":           port_risk,
        "grade_risk":          grade_risk,
        "alloc_df":            alloc_df,
        "ef_df":               ef_df,
        "mix_ef_df":           mix_ef_df,
        "pareto_df":           pareto_df,
        "optimal_pt":          optimal_pt,
        "scen_df":             scen_df,
        "stress_sweep_df":     stress_sweep_df,
        "stress_heatmap_df":   stress_heatmap_df,
        "strat_df":            strat_df,
        "rebal_plan":          rebal_plan,
        "monthly_rebal_df":    monthly_rebal_df,
        "strategic_recs":      strategic_recs,
        "GOVERNANCE_SCORECARD":GOVERNANCE_SCORECARD,
        "RISK_APPETITE":       RISK_APPETITE,
        "PORTFOLIO_TARGETS":   PORTFOLIO_TARGETS,
        "profitability_segments": profitability_segments,
        "concentration_reports":  concentration_reports,
        "segment_opt_results":    segment_opt_results,
        "total_ni_uplift":     total_ni_uplift,
        "resilience_score":    resilience_score,
        "breaking_pd_mult":    breaking_pd_mult,
        "cap_alloc":           cap_alloc,
        "EXEC_SUMMARY":        EXEC_SUMMARY,
        "PO_DIR":              PO_DIR,
        "OPT_DIR":             OPT_DIR,
        "SIM_DIR":             SIM_DIR,
        "DASH_DIR":            DASH_DIR,
        "RPT_DIR":             RPT_DIR,
        "GOV_DIR":             GOV_DIR,
        "STR_DIR":             STR_DIR,
        "PIP_DIR":             PIP_DIR,
    }
 
 
# artefacts = get_module9_artefacts()  # uncomment to import in Module 10+
print("\n✅  Module 9 orchestrator ready. Call get_module9_artefacts() to export.")


══════════════════════════════════════════════════════════════════════
  STEP 19 — PRODUCTION PORTFOLIO INTELLIGENCE PIPELINE
══════════════════════════════════════════════════════════════════════
10:32:51 | INFO     | Production pipeline saved.
  ✅  Production pipeline: portfolio_optimization/pipelines/portfolio_pipeline.py
  ✅  Streamlit blueprint: portfolio_optimization/pipelines/streamlit_portfolio_blueprint.py

══════════════════════════════════════════════════════════════════════
  MODULE 9 — EXECUTIVE SUMMARY
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 9 — PORTFOLIO OPTIMIZATION & STRATEGIC LENDING INTELLIGENCE  ║
║  EXECUTIVE SUMMARY                                                   ║
║  Prepared for: CRO / CEO / Portfolio Management / Strategy Team      ║
║  Generated: 2026-05-23 10:32                              ║
╠══════════════════════════════════════════════════